# Create Dictionary

11/2024
This program can be used to create source-transform cross-reference look-up dictionaries.  This program expects you to provide a source field dictionary that provide the source file name and a description of the field in a simple json file (field:description,).

If a source field dictionary is not available, then there is code below that will reverse engineer using the CIM fields and descriptions.  You will need to create a source field to cim field xreference, shown in the code below the GUI section.

The way the GUI works is it reads the CIM datasets info for all CIM controlled datasets (i.e. all the ones Xentity manages).  Then you click on the drop-down to pick a dataset, then you click on the Source Def button to get the file that lists the source fields and definitions.   Then click on create button, and the program used the source fields, and tries to match them to the fields listed for the dataset on CIM.  It provides a GUI where you can make adjustments and corrections.  THen, when things seems good, you hit write file button on that gui to create the output source-transform dictionary.  The file is written in the directory where this program resides.  

In [1]:
PySimpleGui_license = "eyypJBMoabWpNOl3bknaNJlDVEHHlTwdZ1S1IW6DIvkyRul5dRmfVnsQbC39BflccriUInsbIPk9xtpFY12TVLuLcp2qVSJ0R6C6IW6zMlTAcQx1MVDYgg26NRzlk31ZNpyGweihTNGUlcjvZKW75uzmZvUcRwl3ctG4xrvfeMWf1vlhbLngR2WrZvXaJVzkaTWo9NuuIojkoyxsL2COJUOuYlWT1WlgRMm7lhyNcK3XQtiTOUimJmKDbg2yU9i7L8CuJYOBYsWU1olvTnG7FbzudyCcIj6CIhkYN3v5bMW1VjhYd5XpgaivL5CDJPDIba2K1fwuY6Wv545uICjMosiGIRi1wPiIQv3kVLzJdAGA9stGZyXZJaJ9RKCeIg6rIhjika32MRTjgii1LgCgJAEIYuXNRolVSQXrN0z7deWFVlkbI7j9oAiGMoDAM0v1MYTzkbv5M1jiAvyIN3CkIwsgIEkGRQhXdNGYVEFleKHDBEpucLm3VmzKI1jQoliOM8DJMXvhMtTFk4vZMljpABybNWSiIXs9IokTVgt6YSW6lusbQWWPRmkTcpmTVFzlcQy8IY6yIVmpponQaMXRJTsLci22RghoZHEZBIjxb62w1ojCYFXlNu0VLImg5alBdXCBILs5ISkal4QGQUWURSkkc1mNVLzBcby9IS6OI7jac5z3LUjHI4x7NRyt4UzpMzCW4yxBNBzYEsibfiQT=4=j497179925aaa900272fdf4f5d325d5ad4cbdfb45e641daaad9c58452b474e7229fc8e4deed55e188f08bf7269bc0ac6b0feedba21122843bf056e88dd7e94bc804608ef5e790dd6d78121096ed39d917fce2f82a489c62658e7011ab82d4f6becd37041102470ae619115e19cc0256a6fdf048408e679eb33214ac970ca20bb43815d9c022747fda695279b523015398a42e812a3523ff36887cae0e889b9d4b686a15fffa42df8b19ef0f3efcc65e90b1892b3787ba75ad69e474f9bfa07390e0cf9730a3158488f598d9ff210c8c0fc088e084cc2c7c08a0c2db18b51aa667dbf2a9246f01ea7ed1350067edd5d84dcf32802a90d07657810be9b3ab6a953ddf593fc67048d79ebb3709eee1aa22ebfab09090d26ed5fb3d80549e06c5194adda22d647946d8fa3387f4996e7d42c4cb83b137f3181db34ac0debcc598d381ebd73324acbc0b56cc3add18eac704ed3130ea17a80224ef9db233ea58e1759ff521ea4c1cf95b334e37e4c0458c51318879c2a2ba59361f9f075e4dd281dccac9f7af70e2f14973bc63758518a37bd56324b4e427693700e66ebc39484c8df5a41a8ea8a7e8946867cb42ec44f0e058f8815af53977eaa182ee60d748afe7467427c885c9598de1caa45427b998a18a60efdb67512049d89c3ddfb844b893efeb59326a7d0dd844dc004ede1092342f6b17269d4890866fd345a9a0b7691caa"

from sodapy import Socrata
import pandas as pd
import json,csv
import os,inspect,sys
from chlorophyll import CodeView
from tkinter import Tk
import tkinter as tk
from tkinter import ttk,Button,Label
from datetime  import datetime
#import PySimpleGUI as sg

from difflib import SequenceMatcher
import pygments.lexers

today = datetime.today()
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', None)
cim_url_query = 'data.colorado.gov'
datasets = None
tody = datetime.today()
#today=f"{tody.year}-{tody.month:02d}-{tody.day:02d}"    
cimDatasets = []
bicHome = "/home/joe/bic_etl"
with Socrata(cim_url_query, None) as client:
    datasets = client.datasets()
    for dataset in datasets:
        if dataset['owner']['display_name'] == 'Colorado Information Marketplace':
            cimDatasets.append(dataset)

print("DONE")

DONE


In [5]:
cimDatasets = []

with Socrata(cim_url_query, None) as client:
    datasets = client.datasets()
    for dataset in datasets:
 #       if dataset['owner']['display_name'] == 'Business Inteligence Center of CO':
            cimDatasets.append(dataset)


In [8]:
cimDatasets[0]['owner']

{'id': 'ipka-6wxp',
 'user_type': 'interactive',
 'display_name': 'Kalpana Hanumantharao'}

In [10]:
hist={}
for ds in cimDatasets:
    id=ds['owner']['id']
    name=ds['owner']['display_name']
    hist[id]=name

In [11]:
hist

{'ipka-6wxp': 'Kalpana Hanumantharao',
 'f5d2-2476': 'brian.rohde@state.co.us',
 '8cet-tw9x': 'Colorado Information Marketplace',
 'g32q-fx7y': 'Division of Water Resources',
 'azfj-qb4j': 'Business Intelligence Center of CO',
 'qnx2-ugde': 'JL',
 '3afb-cw7i': 'TranxxQD',
 'ec4k-me69': 'CAE Sanctions',
 'hqpp-wzx3': 'DORA Dental View All Status',
 '6b87-a6rz': 'NathanFlickinger',
 'cbe5-4jhr': 'E Brown',
 '6zdn-4vnv': 'Tina',
 'yw87-vj4e': 'Anthony',
 'wrwr-md9b': 'nate',
 'wxtx-nfsz': 'Mike',
 'j652-bimx': 'LauraN',
 's35q-v5ad': 'shwilliam',
 'w94g-2me8': 'sallymay',
 'faxe-xcp6': 'Progressive Therapy Education',
 'm5uq-aixu': 'thyatt',
 'fkgi-uudt': 'Rosemary',
 '7q7g-umva': 'Chris',
 'djsa-3u43': 'Dan Condren',
 'nppa-79jf': 'ErinJ',
 'tvhs-j5ha': 'BonaguiA',
 'vmnb-8e5v': 'sammontoia',
 'hyp3-nand': 'Carrie',
 '9hj5-f632': 'Mike Eytel',
 '547v-7eh8': 'JoeBiggDogg',
 'emd3-ndxw': 'Marielle',
 '3efi-dsiy': 'DSwenson',
 'kubn-9q24': 'Stephanie',
 '3c6n-k2m6': 'Karie',
 'yfei-vkzy': '

In [4]:
#  These are the CDOS datasets put into a menu.  You will need to add new non-CDOS titles here

groupMenu = [
      'Durable Medical Equipment Suppliers in Colorado',
      'Current Notaries in Colorado',
      'Uniform Commercial Code (UCC) Collateral Information in Colorado',
       'Uniform Commercial Code (UCC) Debtor Information in Colorado',
       'Uniform Commercial Code (UCC) Filing Information in Colorado',
       'Secured Party Information in Colorado',
      'Business Entities in Colorado',
       'Business Entity Transaction History',
       'Trademarks for Businesses in Colorado',
       'Trade Names for Businesses in Colorado',
       'Master List in Colorado',
       'Federal Tax-Exempt Subsection Codes in Colorado',
       'Registration for Charities, Paid Solicitors, Professional Fundraising Consultants, and for-profit Public Benefit Corporations in Colorado',
       'Charitable Organizations’ Offices in Colorado',
       'Other State Solicitation of Charities’ Registrants in Colorado',
       'Charitable Purpose of the Charity in Colorado',
       'Paid Solicitor Solicitation Notices in Colorado',
       'Campaign Reports for Solicitation Notices to Charities in Colorado',
       'Solicitation Campaign Supervisors Listed on Solicitation Notices in Colorado',
       'Charity Extension Requests',
       'Persons Associated with Charitable Organizations, Paid Solicitors, and Professional Fundraising Consultants in Colorado',
       'Other Names a Registered Entity Uses to Solicit Contributions in Colorado',
       'Paid Solicitors Disclosed on Charity Registration Forms in Colorado',
       'Charitable Solicitation Call Center Locations in Colorado',
       'Charities Solicitation Type by Solicitation in Colorado',
       'Communication Methods Used in Solicitation Campaigns in Colorado',
       'Directory of Lobbyists in Colorado',
       'Directory of Lobbyist Clients in Colorado',
       'Expenses for Lobbyists in Colorado',
       'Characterization of Lobbyist Clients in Colorado',
       'Subcontractors for Lobbyists in Colorado',
       'Bill Information and Position with Income of Lobbyist in Colorado'
       ]

##  This loads the source data file name and the transform program... The transform program is displayed so you can compare
##  what the source fields were changed to for CIM (i.e. it give the old source-transform xreference).  

lookupInfo = {
'Solicitation Campaign Supervisors Listed on Solicitation Notices in Colorado':['persons_sol_ntcs.tsv','sol_campaign_supervisors.js'],
'Paid Solicitor Solicitation Notices in Colorado':['sol_ntcs.tsv','sol_notices.js'],
'Campaign Reports for Solicitation Notices to Charities in Colorado':['sn_cmpgn_rpts.tsv','campaign_reports.js'],
'Charitable Organizations’ Offices in Colorado':['offices.tsv','charity_offices.js'],
'Charitable Purpose of the Charity in Colorado':['purpose.tsv','purpose.js'],
'Charitable Solicitation Call Center Locations in Colorado':['sol_ntcs_locs.tsv','sol_ntcs_loc.js'],
'Charities Solicitation Type by Solicitation in Colorado':['sol_typ_sol_ntcs.tsv','sol_typ_sol_ntcs.js'],
'Charity Extension Requests':['char_orgs_ext.tsv','char_orgs.js'],
'Communication Methods Used in Solicitation Campaigns in Colorado':['sol_typ_entity.tsv','sol_typ_entity.js'],
'Federal Tax-Exempt Subsection Codes in Colorado':['tax_exempt_cds.tsv','tax_exempt_cds.js'],
'Other Names a Registered Entity Uses to Solicit Contributions in Colorado':['doing_bus_as.tsv','doing_bus_as.js'],
'Other State Solicitation of Charities’ Registrants in Colorado':['states.tsv','other_state_solicitations.js'],
'Paid Solicitor Solicitation Notices in Colorado':['sol_ntcs.tsv','sol_notices.js'],
'Master List in Colorado':['masterlist.tsv','/home/joe/bic_etl/cdos/business/business/scripts/masterlist.js'],
'Paid Solicitors Disclosed on Charity Registration Forms in Colorado':['char_orgs_sol.tsv','char_orgs_sol.js'],
'Persons Associated with Charitable Organizations, Paid Solicitors, and Professional Fundraising Consultants in Colorado':['persons_entity.tsv','persons_entity.js'],
'Registration of Charities, Paid Solicitors, Professional Fundraising Consultants, and for-profit Public Benefit Corporations in Colorado':['reg_finan.tsv','reg_finan.js']
}

## Functions

In [5]:
def writeFieldFileN3(w4x4,title,values,targDir,datasetCharId):
    ''' Read the Form created for mapping Source Fields to Transformed Fields, 
        write out to a json file, called  w4x4_fields.json
        This checks all the fiels for proper mapping b/w source fields and 
        transformed fields
    '''
    global mapped,dct
    mapped = {}
    mapped[w4x4] = {}
    # mapped["info"] = {"4x4":w4x4,"Title":title}
    # mapped["data"] = {}
    
    xrefs = {}
    notes = {}
    dates = {}
    hold = {}
    oFCount=0
    print("OH YEah... WRITING SOME XREFS")
    print("OH YEah... WRITING SOME XREFS")
    
###    print(values)

    print("DONE WITH VALUES")
    print("DONE WITH VALUES")
    
## Get Values read from Field List Form, put them into a dictionary for output 
## as a JSON

#'SOURCE:Entity Id': 'Entity Id', 'LISTBOX:Entity Id': 'entityId', 'DESC:Entity Id': 'Entity ID', 'NOTES:Entity Id': '', 'DATE:Entity Id': '2024-1-18', 'SOURCE:Document Id': 'Document Id', 'LISTBOX:Document Id': 'documentId', 'DESC:Document Id': 'Document Identifier Number', 'NOTES:Document Id': '', 'DATE:Document Id': '2024-1-18', 
    sourceCounts={}
    hist={}
    for key,val in values.items():
        key=str(key)
        spl=key.split(":")
        nn=spl[2]
        typ=spl[0]
        trCol=spl[1]
        if nn not in hist:
            hist[nn]={}
            
        hist[nn][typ]=val
    dupeCols={}
    for nn in hist.keys():
         if 'SOURCE' in hist[nn]:
            indx = hist[nn]['SOURCE']
            xref = hist[nn]['LISTBOX']
            desc = hist[nn]['DESC']
            date = hist[nn]['DATE']
            
            if 'DUPE' in hist[nn]:
                dupe = hist[nn]['DUPE']
            else:
                dupe = False
        
            if dupe:
                if indx not in dupeCols:
                    dupeCols[indx]=0
                dupeCols[indx]+=1
                num=dupeCols[indx]
                indx+=f"_{num}"
            else:
                oFCount+=1   # count of unique Source Fields
            mapped[w4x4][indx]={}
            mapped[w4x4][indx]['xref'] = xref 
            mapped[w4x4][indx]['desc'] = desc
            mapped[w4x4][indx]['date'] = date
            mapped[w4x4][indx]['dupe'] = dupe
            if dupe:
                mapped[w4x4][indx]['srcflag']=1
                
            
         elif 'SOURCEC' in hist[nn]: 
            indx = hist[nn]['SOURCEC']
            if len(indx) > 0:
                xref = hist[nn]['TRANSC']
                desc = hist[nn]['DESCC']
                date = hist[nn]['DATEC']
                if 'DUPEC' in hist[nn]:
                    dupe = hist[nn]['DUPEC']
                else:
                    dupe = False
                if dupe:
                    if indx not in dupeCols:
                        dupeCols[indx]=0
                    dupeCols[indx]+=1
                    num=dupeCols[indx]
                    indx+=f"_{num}"
    
                mapped[w4x4][indx]={}
                mapped[w4x4][indx]['xref'] = xref 
                mapped[w4x4][indx]['desc'] = desc
                mapped[w4x4][indx]['date'] = date
                mapped[w4x4][indx]['dupe'] = dupe
                if dupe:
                    mapped[w4x4][indx]['srcflag']=1
                print(f"CC {len(indx)}  {indx}  {xref}  {desc}  {date}  {dupe}")
                
    print("Check custom fields")
    print("Check custom fields")
    print("Check custom fields")

#GOT SOME  18 {'SOURCEC': 'Extra Field', 'TRANSC': 'goodStuff', 'DESCC': 'Something', 'DATEC': '2024-12-10', 'DUPEC': True}

    print("LEN MAPPED 1",len(mapped[w4x4]))
    print("LEN MAPPED 1",len(mapped[w4x4]))
    
    try: 
        tmp={}
        good=True
        stringBad=""
        tFCount=0
        
        stringMap=""
##  Check mapped fields for duplicates
        for src,dct in mapped[w4x4].items():
            trn=dct['xref']
            stringMap+= f"{src}  ->  {trn}\n"
            if trn not in tmp:
                tmp[trn]=0
            tmp[trn]+=1
        for oF,count in tmp.items():
            if count > 1:
                print(f"Roh-Roh... Mulitple listings for Transform Field {oF}   Found {count} times")
                stringBad+=f"Roh-Roh... Mulitple listings for Transform Field {oF}   Found {count} times\n"
                good=False
            else:
                tFCount+=1
                
    ##  Used to close tkinter window               
        def close():
            popup.destroy()
            popup.quit()


        if good:
            
            stringAll = f"Dataset {w4x4}\n Source Fields {oFCount}\nTransformed Fields {tFCount}\n\n\n"
            stringAll+=stringMap
            tkPopScrollSimple(stringAll)
   #         sg.popup(stringAll)
        else:
            stringAll = "Errors were Found... file will NOT be written\n\n"
            stringAll+= "Please FIX ERRORS , then retry writing the file\n\n"
            stringAll+= "Transformed Field xrefed to Multiple Source Fields and/or Transformed Fields are set to NONE\n\n"
            stringAll+= " Transformed Field  ->  Source Field\n\n"
            stringAll+= stringBad
            print("BAD BAD BAD",stringAll)
            popup = tk.Tk()
            popup.wm_title("Field List Error")
            label = Label(popup, text=stringAll,font=('Courier',20),justify="left", relief="groove")
            label.configure(bg="pink")
            label.pack(side="top", fill="x", pady=10)
            # B1 = ttk.Button(popup, text="Okay", command = popup.destroy)
            # B1.pack()
            my_button= Button(popup, text= "OK", font=('Courier',25),borderwidth=2, command= close)
            my_button.pack(pady=20)
            popup.mainloop()
            return

        outFile = os.path.join(targDir,f"{datasetCharId}_{w4x4}_src_trns_xrefs.json")
        print("WRITING XREF FIELDS TO ",outFile)
        with open(outFile,"w") as jfile:
            jfile.write(json.dumps(mapped))
    except Exception as err:
         exceptionLog(err,inspect.currentframe().f_code.co_name)
         print("ERROR ",err)

################################


def showSrcCols(title,data):
## I believe this gui will show the source field descriptions 
    print("SHOW RC COLS ",data)
    header=["Field Name"]
    colWidths = [15,30]
    rows=[]
    colors = ["#D5F5E3","#A3E4D7"]
    for nn,dat in enumerate(data):
        row = []
        colr = colors[nn%2]
        row.append(sg.Text(dat,font='Courier 15',justification="right",size=(30,1)))
        rows.append([sg.Frame("",[row],size=(600,50) )])
        
   
   
    layout = [
        [sg.Text(f"CIM Column Descriptions for {title}",font="CENTAUR 15 bold")],
        [sg.Button("Quit",font="CENTAUR 15 bold")],
        [sg.Column(rows,scrollable=True,pad=(20,20),size=(600,400))]
        # [sg.Table(values=data,headings=header, background_color=colors[0],alternating_row_color=colors[1],
        #           num_rows=len(data),row_height=50,vertical_scroll_only=False,max_col_width=60,
        #           enable_events=True, key='-TRANSDESC-',col_widths=colWidths,border_width=6,
        #           font="CENTAUR 10 bold")],
    ]
    sg.theme("LightBrown6")    
    window = sg.Window('Source Data Columns', layout, finalize=True)

###############################

def createSrcTransFieldXrefReverse(w4x4,trans,srcDatFile,targDir):
    '''
    This function creates a Cross Reference List between the 
    Source Field List and Transform Field list for a dataset
    A form will be created and seeded with the Source Field list and 
    Descriptions found in the srcDefFile.  This will thne use the trans
    column listsing for the Transformed Fields to  guess a best match
    for each Source Field.  Users will then be able to fix the guesses 
    for the Transformed Fields.
    The Form allows users to create a new Transform Field if needed.
    The Form also allows for the creation of totally NEW Source and Transform
    listings    
    '''
    today = datetime.today()
    todayDate=f"{today.year}-{today.month}-{today.day}"    
    tmp = {}
    
# create a dictionary of the descripitons of the transform fields for easy lookup
## If there is a source definition file, use it to seed the defintion/descriptions
    if len(srcDatFile) > 4:
        # fin = open(srcDatFile,"r")
        # tmp=json.load(fin)
        # orig=tmp.keys()
        df = getFile("Local",srcDatFile) 
        orig=list(df.columns) 
    else:
        # for nn,fld in enumerate(fields[w4x4]['cim']):
        #     tmp[fld] = fields[cim4x4]['description'][nn]
        sg.popup("No Source Data File Found")
        return "  "
    tmpT2O = {}
    print("ORIG ",orig)
    print("TRNS ",trans)

## Get most likely Match of Original Column to Transformed Columnm
    for col in trans.keys():
        cc = getMostLike(col,orig)
        tmpT2O[col]=cc
        
    # print("TMPO2T ",tmpO2T)
    # print("TMP ",tmp)
    # print("ORIG ",orig)
    col1 = []
    col2 = []
    col3 = []
    rows = []
    origTmp = orig.copy()
    row=[]
    rowC=[]
    
 
    
 #   ["Source Field","Transformed Field","Description","Date","Notes"]
    row.append(sg.Text("Source Field",font='Courier 10 bold ',text_color="black",justification="left",size=(30,1))) 
    rowC.append(sg.Text("Source Field",font='Courier 10 bold ',text_color="black",key="HC1",justification="center",size=(30,1))) 

    row.append(sg.Text("Transformed Field",font='Courier 10 bold ',text_color="black",justification="left",size=(30,1)))    
    rowC.append(sg.Text("Transformed Field",font='Courier 10 bold ',text_color="black",key="HC2",justification="center",size=(30,1)))    

    row.append(sg.Text("Description",font='Courier 10 bold ',text_color="black",justification="left",size=(30,1)))    
    rowC.append(sg.Text("Description",font='Courier 10 bold ',text_color="black",key="HC3",justification="center",size=(30,1)))    

    row.append(sg.Text("Notes",font='Courier 10 bold ',text_color="black",justification="left",size=(30,1)))    
    rowC.append(sg.Text("Notes",font='Courier 10 bold ',text_color="black",key="HC4",justification="center",size=(30,1)))    

    row.append(sg.Text("Change Date",font='Courier 10 bold ',text_color="black",justification="left",size=(15,1)))   
    rowC.append(sg.Text("Change Date",font='Courier 10 bold ',text_color="black",key="HC5",justification="center",size=(15,1)))   

    row.append(sg.Text("Source Duplicate",font='Courier 10 bold ',text_color="black",justification="left",size=(15,1)))   
    rowC.append(sg.Text("Source Duplicate",font='Courier 10 bold ',text_color="black",key="HC5",justification="center",size=(15,1)))   

    
    #   frameCustHead = sg.Frame("",[row],key="FRAME:CUSTHEAD",visible=False)
    rows.append([sg.Frame("",[row],key="HEADER")])
    
    for count,col in enumerate(list(trans.keys())):
       #   print("COL ",col)
        
          row=[]
#         row.append(sg.Input(col,key=f"SOURCE:{col}",font='Courier 15 bold ',size=(30,1)))
##  put the most likely matching string on top of list, so it appears as first
##  option ins combo box
          origTmp = reorderList(tmpT2O[col],sorted(origTmp))
##  Add NONE as an option to account for No Matches in terms of a new field
          a=origTmp.copy()
          a.append("NONE")    
          if col not in tmpT2O:
                print(f"MISS MISS in function createSrcTransFieldXrefReverse:  tmpT2O :{col}:")
                
          # if col not in tmp:
          #       print(f"MISS MISS tmp :{col}:")
          
          row.append(sg.Combo(a,default_value=tmpT2O[col],enable_events=True,  
                                font='Courier 15 bold ',key=f"SOURCE:{col}:{count}",
                         #       select_mode="LISTBOX_SELECT_MODE_SINGLE",
                                size=(30,6)))
          row.append(sg.Input(col,key=f"LISTBOX:{col}:{count}",font='Courier 15 bold ',size=(30,1)))
          row.append(sg.Multiline(trans[col],font='Courier 10',key=f"DESC:{col}:{count}",size=(30,2)))
          row.append(sg.Multiline("",font='Courier 10',size=(30,2),key=f"NOTES:{col}:{count}"))
          row.append(sg.Input(todayDate,font='Courier 10 bold ',size=(15,1),key=f"DATE:{col}:{count}"))
          row.append(sg.Check("Duplicate ",key=f"DUPE:{col}:{count}"))
        
          rows.append([sg.Frame("",[row],size=(1800,50))])
        
    rowsC = []
    rowsC.append([sg.Frame("",[rowC],key="someFrame",visible=False)])

##  Add custom fields, make them invisible until needed, one by one    
    for nn in range(1,15):
          row=[]
          count+=1
          row.append(sg.Input("",
                                font='Courier 15 bold ',key=f"SOURCEC:CUST:{count}",visible=True,
                         #       select_mode="LISTBOX_SELECT_MODE_SINGLE",visible=False,
                                size=(21,1),enable_events=False))
          row.append(sg.Input("",font='Courier 15 bold ',visible=True,key=f"TRANSC:CUST:{count}",size=(31,1)))
          row.append(sg.Input("",font='Courier 15',size=(31,2),visible=True,key=f"DESCC:CUST:{count}"))
          row.append(sg.Multiline("",font='Courier 15',size=(31,1),visible=True,key=f"NOTESC:CUST:{count}"))
          row.append(sg.Input(todayDate,font='Courier 15 bold ',visible=True,size=(11,1),key=f"DATEC:CUST:{count}"))
          row.append(sg.Check("Duplicate ",key=f"DUPEC:CUST:{count}"))
        
          rowsC.append([sg.Frame("",[row],visible = True,key=f"FRAME:CUST:{count}")])
    rows.append([sg.Frame("Custom Fields",[rowsC])])
        
        
    # layout = [[sg.Button("Quit"),sg.Button("Add XREF Field")],
    #            [sg.Button("Write XREF File")],
    #            [sg.Column(rows,scrollable=True,pad=(20,20))],
    #            [sg.Frame("Custom Fields",rowsC,font='Courier 15 bold ',visible=False,relief="sunken",key="secondFrame")]]
    row=[]
    rowsl=[]
    row.append(sg.Button("Quit"))
#    row.append(sg.Button("Add XREF Field"))
    rowsl.append(row)
    row=[]
    row.append(sg.Button("Write XREF File"))
    rowsl.append(row)
    row=[]
    nsz = len(rows)*50
    print("SIZE SIZE SIZE ",nsz)
    row.append(sg.Column(rows,scrollable=True,pad=(20,20),size=(1800,1200))) 
#    row.append(sg.Sizer(1800,400))

    rowsl.append(row)
#    row=[]
#    row.append(sg.Frame("Custom Fields",rowsC,font='Courier 15 bold ',visible=False,relief="sunken",key="secondFrame"))
#    rowsl.append(row)
    layout = [[sg.Column(rowsl,scrollable=True)]]
        
        
    sg.theme("LightBrown6")     
    window2 = sg.Window("Log Summary",layout,finalize=True,resizable=True)
    a = window2.CurrentLocation()
    screen_width, screen_height = window2.get_screen_dimensions()
    win_width, win_height = window2.size
    x, y = (screen_width - win_width)//2, (screen_height - win_height)//2
    x=200
    y=200
    window2.move(x, y)
      
    return origTmp

###############################################################



def showTransDesc(title,data):
## I believe this gui will show the source field descriptions 
    header=["Field Name","Description"]
    colWidths = [15,30]
    rows=[]
    colors = ["#D5F5E3","#A3E4D7"]
    for nn,dat in enumerate(data):
        row = []
        colr = colors[nn%2]
        row.append(sg.Text(dat[0],font='Courier 15',justification="right",size=(30,1)))
        row.append(sg.Multiline(dat[1],font='Courier 15',justification="left",size=(60,2)))
        rows.append([sg.Frame("",[row],size=(600,50)
                             )])
        
   
   
    layout = [
        [sg.Text(f"CIM Column Descriptions for {title}",font="CENTAUR 15 bold")],
        [sg.Button("Quit",font="CENTAUR 15 bold")],
        [sg.Column(rows,scrollable=True,pad=(20,20),size=(600,400))]
        # [sg.Table(values=data,headings=header, background_color=colors[0],alternating_row_color=colors[1],
        #           num_rows=len(data),row_height=50,vertical_scroll_only=False,max_col_width=60,
        #           enable_events=True, key='-TRANSDESC-',col_widths=colWidths,border_width=6,
        #           font="CENTAUR 10 bold")],
    ]
    sg.theme("LightBrown6")    
    window = sg.Window('CIM Field Descriptions', layout, finalize=True)

###############################

def createSrcTransFieldXref(w4x4,trans,srcDefFile,targDir):
    '''
    This function creates a Cross Reference List between the 
    Source Field List and Transform Field list for a dataset
    A form will be created and seeded with the Source Field list and 
    Descriptions found in the srcDefFile.  This will thne use the trans
    column listsing for the Transformed Fields to  guess a best match
    for each Source Field.  Users will then be able to fix the guesses 
    for the Transformed Fields.
    The Form allows users to create a new Transform Field if needed.
    The Form also allows for the creation of totally NEW Source and Transform
    listings    
    '''
    today = datetime.today()
    todayDate=f"{today.year}-{today.month}-{today.day}"    
    tmp = {}
    
# create a dictionary of the descripitons of the transform fields for easy lookup
## If there is a source definition file, use it to seed the defintion/descriptions
    if len(srcDefFile) > 4:
        fin = open(srcDefFile,"r")
        tmp=json.load(fin)
        orig=tmp.keys()
    else:
        # for nn,fld in enumerate(fields[w4x4]['cim']):
        #     tmp[fld] = fields[cim4x4]['description'][nn]
        sg.popup("No Source Definition File Found")
        return "  "
    tmpO2T = {}
    print("ORIG ",orig)
    print("TRNS ",trans)
    for col in orig:
        cc = getMostLike(col,trans)
        tmpO2T[col]=cc
        
    # print("TMPO2T ",tmpO2T)
    # print("TMP ",tmp)
    # print("ORIG ",orig)
    col1 = []
    col2 = []
    col3 = []
    rows = []
    transTmp = trans.copy()
    row=[]
    rowC=[]
    
 
    
 #   ["Source Field","Transformed Field","Description","Date","Notes"]
    row.append(sg.Text("Source Field",font='Courier 10 bold ',text_color="black",justification="left",size=(30,1))) 
    rowC.append(sg.Text("Source Field",font='Courier 10 bold ',text_color="black",key="HC1",justification="center",size=(30,1))) 

    row.append(sg.Text("Transformed Field",font='Courier 10 bold ',text_color="black",justification="left",size=(30,1)))    
    rowC.append(sg.Text("Transformed Field",font='Courier 10 bold ',text_color="black",key="HC2",justification="center",size=(30,1)))    

    row.append(sg.Text("Description",font='Courier 10 bold ',text_color="black",justification="left",size=(30,1)))    
    rowC.append(sg.Text("Description",font='Courier 10 bold ',text_color="black",key="HC3",justification="center",size=(30,1)))    

    row.append(sg.Text("Notes",font='Courier 10 bold ',text_color="black",justification="left",size=(30,1)))    
    rowC.append(sg.Text("Notes",font='Courier 10 bold ',text_color="black",key="HC4",justification="center",size=(30,1)))    

    row.append(sg.Text("Change Date",font='Courier 10 bold ',text_color="black",justification="left",size=(15,1)))   
    rowC.append(sg.Text("Change Date",font='Courier 10 bold ',text_color="black",key="HC5",justification="center",size=(15,1)))   

    #   frameCustHead = sg.Frame("",[row],key="FRAME:CUSTHEAD",visible=False)
    rows.append([sg.Frame("",[row],key="HEADER")])
    
    for col in orig:
       #   print("COL ",col)
        
          row=[]
          row.append(sg.Input(col,key=f"SOURCE:{col}",font='Courier 15 bold ',size=(30,1)))
##  put the most likely matching string on top of list, so it appears as first
##  option ins combo box
          transTmp = reorderList(tmpO2T[col],sorted(transTmp))
##  Add NONE as an option to account for No Matches in terms of a new field
          a=transTmp.copy()
          a.append("NONE")    
          if col not in tmpO2T:
                print(f"MISS MISS tmpO2T :{col}:")
                
          if col not in tmp:
                print(f"MISS MISS tmp :{col}:")
          
          row.append(sg.Combo(a,default_value=tmpO2T[col],enable_events=True,  
                                font='Courier 15 bold ',key=f"LISTBOX:{col}",
                         #       select_mode="LISTBOX_SELECT_MODE_SINGLE",
                                size=(30,6)))
          row.append(sg.Multiline(tmp[col],font='Courier 10',key=f"DESC:{col}",size=(30,2)))
          row.append(sg.Multiline("",font='Courier 10',size=(30,2),key=f"NOTES:{col}"))
          row.append(sg.Input(todayDate,font='Courier 10 bold ',size=(15,1),key=f"DATE:{col}"))
        
          rows.append([sg.Frame("",[row],size=(1800,50))])
        
    rowsC = []
    rowsC.append([sg.Frame("",[rowC],key="someFrame",visible=False)])

##  Add custom fields, make them invisible until needed, one by one    
    for nn in range(1,15):
          row=[]
         
          row.append(sg.Input("",
                                font='Courier 15 bold ',key=f"SOURCEC:CUST:{nn}",visible=True,
                         #       select_mode="LISTBOX_SELECT_MODE_SINGLE",visible=False,
                                size=(21,1),enable_events=False))
          row.append(sg.Input("",font='Courier 15 bold ',visible=True,key=f"TRANSC:CUST:{nn}",size=(31,1)))
          row.append(sg.Input("",font='Courier 15',size=(31,2),visible=True,key=f"DESCC:CUST:{nn}"))
          row.append(sg.Multiline("",font='Courier 15',size=(31,1),visible=True,key=f"NOTESC:CUST:{nn}"))
          row.append(sg.Input(todayDate,font='Courier 15 bold ',visible=True,size=(11,1),key=f"DATEC:CUST:{nn}"))
        
          rowsC.append([sg.Frame("",[row],visible = True,key=f"FRAME:CUST:{nn}")])
    rows.append([sg.Frame("Custom Fields",[rowsC])])
        
        
    # layout = [[sg.Button("Quit"),sg.Button("Add XREF Field")],
    #            [sg.Button("Write XREF File")],
    #            [sg.Column(rows,scrollable=True,pad=(20,20))],
    #            [sg.Frame("Custom Fields",rowsC,font='Courier 15 bold ',visible=False,relief="sunken",key="secondFrame")]]
    row=[]
    rowsl=[]
    row.append(sg.Button("Quit"))
#    row.append(sg.Button("Add XREF Field"))
    rowsl.append(row)
    row=[]
    row.append(sg.Button("Write XREF File"))
    rowsl.append(row)
    row=[]
    nsz = len(rows)*50
    print("SIZE SIZE SIZE ",nsz)
    row.append(sg.Column(rows,scrollable=True,pad=(20,20),size=(1800,1200))) 
#    row.append(sg.Sizer(1800,400))

    rowsl.append(row)
#    row=[]
#    row.append(sg.Frame("Custom Fields",rowsC,font='Courier 15 bold ',visible=False,relief="sunken",key="secondFrame"))
#    rowsl.append(row)
    layout = [[sg.Column(rowsl,scrollable=True)]]
        
        
    sg.theme("LightBrown6")     
    window2 = sg.Window("Log Summary",layout,finalize=True,resizable=True)
    a = window2.CurrentLocation()
    screen_width, screen_height = window2.get_screen_dimensions()
    win_width, win_height = window2.size
    x, y = (screen_width - win_width)//2, (screen_height - win_height)//2
    x=200
    y=200
    window2.move(x, y)
      
    return transTmp

###############################################################

def showSrc2TransXrefs(file):
    '''  Display the Source to Transformed Fields Dictionary for the Dataset.  Note you have to read this dictionary in 
         to display it.  This does not view one stored in memory (i.e. the one you just created), this only displays one 
         you have ahd to program read in. 
    '''
    if not os.path.isfile(file):
       sg.popup(f"Source-Transform Dictionary File Does NOT EXIST..")
       return
    todayDate=f"{today.year}-{today.month}-{today.day}"    
    
    fin = open(file,"r")
    info = json.load(fin)
    rows=[]
    row=[]
    row.append(sg.Text("Source Field",font='Courier 15 bold ',background_color="#c2c2d6",text_color="black",justification="left",size=(30,1))) 
    row.append(sg.Text("Transformed Field",font='Courier 15 bold ',background_color="#c2c2d6",text_color="black",justification="left",size=(30,1)))    
    row.append(sg.Text("Description",font='Courier 15 bold ',background_color="#c2c2d6",text_color="black",justification="left",size=(31,1)))    
    row.append(sg.Text("Notes",font='Courier 15 bold ',background_color="#c2c2d6",text_color="black",justification="left",size=(31,1)))    
    row.append(sg.Text("Change Date",font='Courier 15 bold ',background_color="#c2c2d6",text_color="black",justification="left",size=(15,1)))   
    rows.append(row)
    nrows=0
    colors = ["#6699ff","#66ccff"]
    for w4x4,data in info.items():
        for col,defs in data.items():
              nrows+=1
              colr = colors[nrows%2]
              row=[]
              row.append(sg.Input(col,key=f"SOURCE:{col}",font='Courier 15 bold ',background_color=colr,size=(30,1)))
              row.append(sg.Input(defs['xref'],key=f"LISTBOX:{col}",background_color=colr,font='Courier 15 bold ',size=(30,1)))  
              row.append(sg.Multiline(defs['desc'],key=f"DESC:{col}",background_color=colr,font='Courier 15',size=(30,2)))
              row.append(sg.Multiline(defs['notes'],key=f"NOTES:{col}",background_color=colr,font='Courier 15',size=(30,2)))
              row.append(sg.Input(defs['date'],key=f"DATE:{col}",background_color=colr,font='Courier 15 bold ',size=(15,1)))
              rows.append(row)

        for nn in range(1,6):
              row=[]

              row.append(sg.Input("",
                                    font='Courier 15 bold ',key=f"SOURCEC:CUST:{nn}",visible=True,
                             #       select_mode="LISTBOX_SELECT_MODE_SINGLE",visible=False,
                                    size=(21,1),enable_events=False))
              row.append(sg.Input("",font='Courier 15 bold ',visible=True,key=f"TRANSC:CUST:{nn}",size=(31,1)))
              row.append(sg.Multiline("",font='Courier 15',size=(31,2),visible=True,key=f"NOTESC:CUST:{nn}"))
              row.append(sg.Multiline("",font='Courier 15',size=(31,2),visible=True,key=f"NOTESC:CUST:{nn}"))
              row.append(sg.Input(todayDate,font='Courier 15 bold ',visible=True,size=(11,1),key=f"DATEC:CUST:{nn}"))
              rows.append(row)

   
    header = ["Source Field","Transformed Field","Description","Notes","Date"]
    
    # layout = [[sg.Text(f"Source-Transformed Field XREFS",font="CENTAUR 15 bold")],
    # [sg.Text(f"{file}",font="CENTAUR 15 bold")],
    # [sg.Button("Quit"),sg.Button("Add XREF Fields")],
    # [sg.Button("Write XREF File")],  
    # [[sg.Column(rows,scrollable=True)]]
    # ]
    rowsl=[]
    row=[]
    row.append(sg.Text(f"Source-Transformed Field XREFS",font="CENTAUR 15 bold"))
    rowsl.append(row)
    row=[]
    row.append(sg.Text(f"{file}",font="CENTAUR 15 bold"))
    rowsl.append(row)
    row=[]
    row.append(sg.Button("Quit"))
    row.append(sg.Button("Cmpr to Source Data"))
    rowsl.append(row)
    rowsl.append([sg.Button("Write XREF File")])
    rowsl.append([sg.Column(rows,scrollable=True)])
    layout = [[sg.Column(rowsl,scrollable=True)]]
#    sg.theme(colorTheme)     
    window2 = sg.Window("Source-Transform Field XREFS",layout,background_color="white",finalize=True,resizable=True,metadata=info)
    a = window2.CurrentLocation()
    screen_width, screen_height = window2.get_screen_dimensions()
    win_width, win_height = window2.size
    x, y = (screen_width - win_width)//2, (screen_height - win_height)//2
    x=200
    y=200
    window2.move(x, y)
    
########################################
    
def writeFieldFileN2(w4x4,title,values,targDir,datasetCharId):
    ''' Read the Form created for mapping Source Fields to Transformed Fields, 
        write out to a json file, called  w4x4_fields.json
        This checks all the fiels for proper mapping b/w source fields and 
        transformed fields
    '''
    global mapped,dct
    mapped = {}
    mapped[w4x4] = {}
    # mapped["info"] = {"4x4":w4x4,"Title":title}
    # mapped["data"] = {}
    
    xrefs = {}
    notes = {}
    dates = {}
    hold = {}
    print("OH YEah... WRITING SOME XREFS")
#    print(values)
## Get Values read from Field List Form, put them into a dictionary for output 
## as a JSON

#'SOURCE:Entity Id': 'Entity Id', 'LISTBOX:Entity Id': 'entityId', 'DESC:Entity Id': 'Entity ID', 'NOTES:Entity Id': '', 'DATE:Entity Id': '2024-1-18', 'SOURCE:Document Id': 'Document Id', 'LISTBOX:Document Id': 'documentId', 'DESC:Document Id': 'Document Identifier Number', 'NOTES:Document Id': '', 'DATE:Document Id': '2024-1-18', 


    for key,val in values.items():
        key=str(key)
        print("Key,val ",key,val,type(str))
        if isinstance(key,str) and ":" in key:
       #     'LISTBOX:entityId': 'Entity Id', 0: 'Entity Id', 'NOTES:entityId':
            spl = key.split(":")
            val = val.strip()
            spl[1]=spl[1].strip()
            spl[0]=spl[0].strip()
            print("SPL  ",spl)
            if spl[0] == "SOURCE":  
                if spl[1] != "CUST":
                    if spl[1] not in mapped[w4x4]:                        
                        mapped[w4x4][spl[1]] = {}  
              #      mapped[w4x4][spl[1]]["xref"] = val 
                    
            elif spl[0] == "LISTBOX":
                if spl[1] not in mapped[w4x4]:                        
                    mapped[w4x4][spl[1]] = {}  
                mapped[w4x4][spl[1]]["xref"] = val 
            elif spl[0] == "DESC":
                if spl[1] != "CUST":
                   if spl[1] not in mapped[w4x4]:                        
                        mapped[w4x4][spl[1]] = {}  
                   mapped[w4x4][spl[1]]["desc"] = val        
            elif spl[0] == "NOTES":
                if spl[1] != "CUST":
                    if spl[1] not in mapped[w4x4]:                        
                        mapped[w4x4][spl[1]] = {}  
                    mapped[w4x4][spl[1]]["notes"] = val 
            elif spl[0] == "DATE":
                if spl[1] != "CUST":
                    if spl[1] not in mapped[w4x4]:                        
                        mapped[w4x4][spl[1]] = {}  
                    mapped[w4x4][spl[1]]["date"] = val 
                    
            elif spl[1] == "CUST" and spl[0] in ["SOURCEC","TRANSC","DESCC","NOTESC","DATEC"] and len(val) > 0:
                print("CUST STUFF",spl)
                nn = spl[2]
                if nn not in hold:
                    hold[nn] = {}
                hold[nn][spl[0]] = val
                
    print("Check custom fields")
#    print(hold)
##  Add any custom Fields to mapped
    for k,v in hold.items():
        if "SOURCEC" in v and len(v["SOURCEC"]) > 0 :
            mapped[w4x4][v["SOURCEC"]] = {}
            mapped[w4x4][v["SOURCEC"]]["xref"]=v["TRANSC"]
            if "NOTESC" in v:
                mapped[w4x4][v["SOURCEC"]]["notes"]=v["NOTESC"]
            else:
                mapped[w4x4][v["SOURCEC"]]["notes"]=""
            
            if "DESCC" in v:
                mapped[w4x4][v["SOURCEC"]]["desc"]=v["DESCC"]
            else:
                mapped[w4x4][v["SOURCEC"]]["desc"]=""
                
            mapped[w4x4][v["SOURCEC"]]["date"]=v["DATEC"]
    print("MAPPED ",mapped) 

    try: 
##  Check mapped fields for duplicates
        statsT2O = {}
        statsO = {}
        oFCount=0
        tFCount=0
        stringMap = ""
        stringBad = ""
        
#        for  w4x4,dct in mapped.items():
        dct=mapped
        print("4X4 4X4 4X4 ",w4x4)
        print("ITEMS ",dct[w4x4].items())
        print("S20 ",statsT2O)
        for oF,info in dct[w4x4].items():
             print("KEY ",oF,info)
             tF = info['xref']
             print("MAPPING ",oF,":",tF)
             if tF not in statsT2O:
                statsT2O[tF] = []
                statsT2O[tF].append(oF)
                print("SS ",type(statsT2O))
                if oF not in statsO:
                    statsO[oF] = 0
                statsO[oF]+=1
        print("DONE MAPPING")   
        print("Checking Transformed fields")
        print(statsT2O.items())
    ## Check that Transformed Fields are only being referenced by 1 Original Field            
        for tF,xrfs in statsT2O.items():
            print("CHECKING ",tF,xrfs)
            if len(xrfs) > 1:
                print(f"{tF} too many X-Refs")
                for val in xrfs:
                    stringBad+= f" {tF} -> {val}\n"
                stringBad+=f"\n"
            elif tF == "NONE":
                    stringBad+="Transformed Fields is set to NONE... Please set a Legitimate X-ref\n"
                    stringBad+=f"  {tF} -> {xrfs[0]}\n\n"
            else:
                stringMap+= f" {xrfs[0]} -> {tF}\n"
                tFCount+=1

        for oF,count in statsO.items():
            if count > 1:
                print(f"Roh-Roh... Mulitple listings for {oF}   Found {count} times")
            else:
                oFCount+=1
    ##  Used to close tkinter window               
        def close():
            popup.destroy()
            popup.quit()


        if len(stringBad) == 0:
            
            stringAll = f"Dataset {w4x4}\n Source Fields {oFCount}\nTransformed Fields {tFCount}\n\n\n"
            stringAll+=stringMap
            tkPopScrollSimple(stringAll)
   #         sg.popup(stringAll)
        else:
            stringAll = "Errors were Found... file will NOT be written\n\n"
            stringAll+= "Please FIX ERRORS , then retry writing the file\n\n"
            stringAll+= "Transformed Field xrefed to Multiple Source Fields and/or Transformed Fields are set to NONE\n\n"
            stringAll+= " Transformed Field  ->  Source Field\n\n"
            stringAll+= stringBad
            print("BAD BAD BAD",stringAll)
            popup = tk.Tk()
            popup.wm_title("Field List Error")
            label = Label(popup, text=stringAll,font=('Courier',20),justify="left", relief="groove")
            label.configure(bg="pink")
            label.pack(side="top", fill="x", pady=10)
            # B1 = ttk.Button(popup, text="Okay", command = popup.destroy)
            # B1.pack()
            my_button= Button(popup, text= "OK", font=('Courier',25),borderwidth=2, command= close)
            my_button.pack(pady=20)
            popup.mainloop()
            return

        outFile = os.path.join(targDir,f"{datasetCharId}_{w4x4}_src_trns_xrefs.json")
        print("WRITING XREF FIELDS TO ",outFile)
        with open(outFile,"w") as jfile:
            jfile.write(json.dumps(mapped))
    except Exception as err:
         exceptionLog(err,inspect.currentframe().f_code.co_name)
         print("ERROR ",err)

################################

def tkPopScrollSimple(string):
    def close():
       root.destroy()
       root.quit()

    root = tk.Tk()
    root.resizable(False, False)
    root.title("Scrollbar Widget Example")

    # apply the grid layout
    root.grid_columnconfigure(0, weight=1)
    root.grid_rowconfigure(0, weight=1)

    # create the text widget
    text = tk.Text(root, height=10)
    text.grid(row=0, column=0, sticky=tk.EW)

    # create a scrollbar widget and set its command to the text widget
    scrollbar = ttk.Scrollbar(root, orient='vertical', command=text.yview)
    scrollbar.grid(row=0, column=1, sticky=tk.NS)

    #  communicate back to the scrollbar
    text['yscrollcommand'] = scrollbar.set

    # add sample text to the text widget to show the screen
    # for i in range(1,50):
    #     position = f'{i}.0'
    for nn,val in enumerate(string.split("\n")):
       m = float(nn+1)
       text.insert(f"{m}",f"{val}\n")

    my_button= Button(root, text= "OK", font=('Courier',15),
    borderwidth=2, command= close)
    my_button.grid(row=nn+2, column=0, sticky=tk.EW)
    
    root.mainloop()
        

################################
def getTransCols(title,datasets):
##  Get the CIM Metadata for the input (title) dataset
    for dat in datasets:
        if dat["resource"]["name"] == title:
           break
    cols = dat["resource"]["columns_name"]
    desc = dat["resource"]["columns_description"]
    w4x4 = dat["resource"]["id"]
    
    return cols,desc,w4x4

##############################################

def getFile(how,file):
    ''' This reads in a local file.  how indicates if it is local (Local) or web (Fetch) file.  For local
    files, it will look at the file extensions (tsv,csv,xlsx,xls) and try to read it correctly. FOr tsv,csv if that 
    failes, it will then attempt to read it using encoding=latin. This would not be an issue for xlsx and xls files 
    '''
    print("Getting file ",file,how)
    if how == "Local":
        if file[-3:].lower() == "tsv":
            delim = "\t"
            df=pd.read_csv(file,encoding="latin",delimiter=delim)
        elif file[-4:].lower() == "xlsx":
            df=pd.read_excel(file,engine="openpyxl")
        elif file[-3:].lower() == "xls":
            df=pd.read_excel(file)
        else:
            try:
                df=pd.read_csv(file)
            except Exception as err:
                print("Error, trying with encoding=latin")
                df=pd.read_csv(file,encoding="latin")
    elif how == "Fetch":
        df=pd.read_csv(file)

#    stats = dfAnalyze(df)
        
    return df

##############################################

def getMostLike(a:str,b:list):
    '''Searchs a list of strings for the value that is most like input the string a
       Returns the string most like a
       
       col = getMostLike("someString",someList)
    '''
    rmax = SequenceMatcher(None,a.lower(),b[0].lower()).ratio()
    column=b[0]
    if len(b) > 1:
        for col in b[1:]:
            r=SequenceMatcher(None,a.lower(),col.lower()).ratio()
            if r > rmax:
                rmax=r
                column=col
    return column

################################

def reorderList(string:str,a:list):
    ''' If string exists in list a, it will be moved to the top of the list '''
    b = a.copy()
    if string in b:
        ai = b.index(string)
        b.pop(ai)
        c = [string]
        c[1:] = b
    else:
        c=a.copy()
        
    return c

#####################################

def extractDecode(inputDict):
    ''' Input the listing for 1 dataset out of the run_etl.json dataset files and it will decode the info in the 
        extract section
    ''' 
   # print(inputDict)
    pgm=""
    datasetCharId=""
    localFile=""
    localDir = ""
    extFlag=0
    if "language" in inputDict and inputDict["language"] == "node":
        pgm = f"node {bicHome}{inputDict['file']} "
        nflag=0
        if "options" in inputDict:
            for opts in inputDict["options"]:
                pgm+=opts + " "
                if opts[0:2] == "-f" and inputDict["file"].find("sftp_extract.js") > -1:
                    spl = opts.split("/")
                    if nflag == 0:
                       localFile=spl[-1]
                    spl = spl[1].split(".")
                    datasetCharId = spl[0]
                   
                elif opts[0:2] == "-n" and inputDict["file"].find("sftp_extract.js") > -1:
                    nflag=1
                    localFile=opts[3:]
                elif opts[0:2] == "-o" and inputDict["file"].find("sftp_extract.js") > -1:
                    spl=opts.split(" ")
                    localDir=spl[1]
                elif opts[0:2] == "-a" and inputDict["file"].find("sftp_extract.js") > -1:
                    ext = opts[3:]
                    extFlag=1
                elif opts[0:2] == "-f" and inputDict["file"].find("request_url.js") > -1:
                    spl=opts.split(" ")
                    localFile=spl[1]
    if extFlag == 1:
        period = localFile.find(".")
 #       print(period,localFile,inputDict)
        localFile = localFile[:period] + ext
    localFile = f"{localDir}{localFile}"
    
    return pgm,datasetCharId,localFile

##########################################

def getExtention(title):
    '''  I have a file that lists all the file names for the cdos nonprofit datasets (included in the repo) that 
         lists the dataset and the base file name.  This base file name is used to create the name of the output 
         source-transform cross-reference, which name is    ext_4x4_src_trn_xrefs.json, where ext is in this file and 
         4x4 is extracted from the CIM data via api
    '''
    fin = open("/home/joe/work/CDOS/nonprofit/defs/xrefs.tsv")
    lines = fin.readlines()
    sourceFieldDicts = {}
    for line in lines[1:]:
        a = line.split("\t")
        dictn = a[0].strip()
        tit = a[1].strip()
        if tit == title:
           print(f"ds :{tit}:")
           return dictn[:-4]
    return ""

############################################# 

def exceptionLog(exception,funCall):
  exception_message = str(exception)
  exception_type, exception_object, exception_traceback = sys.exc_info()
  filename = os.path.split(exception_traceback.tb_frame.f_code.co_filename)[1]

#############################################

def showFile(file):
    fin = open(file,"r") 
    text = fin.readlines()
    string=""
    for aa in text:
        string+=aa
    root = Tk()
    root.title(file)
    codeview = CodeView(root, lexer=pygments.lexers.JavascriptLexer, color_scheme="monokai")
    codeview.pack(fill="both", expand=True)
    codeview.insert(tk.END,chars=string)
    root.mainloop()

################################################

def whatFile():
    print("GETTING Rando File")
    layout = [[sg.Button("Quits")],
        [sg.Text("",key="-GINFO-",enable_events=True),sg.Text("Input File "),sg.FilesBrowse(button_text="Get File",target="-GINFO-",initial_folder="/home/joe/",font="CENTAUR 15",file_types=[("All Files","*"),("CSV Files","*.csv"),("TSV Files","*.tsv"),("Excel Files","*.xlsx")],enable_events=True,key='-INPUTFILE-')]
    ]
    window = sg.Window('Get File', layout,finalize=True,resizable=True,background_color="#cc4400")
    try: 
        while True:
            event, values = window.read()
            print("EV ",event)
            print(values)
            if event == sg.WIN_CLOSED or event == 'Quits':
                window.close()
                break
            elif event=="-GINFO-":
                file=values['-INPUTFILE-']
    except Exception as err:
        print("ERROR in whatFile",err)

    return ""

def compare2Source(info,file):
    print("GETTING Rando File")
    if len(file) > 0:
        df = getFile("Local",file) 
        colsData = list(df.columns)
        notindict=0
        notindata=0
        print("\n-----------------")
        
        print("\nColumns in SRC-TRANSFORM Dictionary not found in DATA") 
        
        for w,dct in info.items():
          
            for col,dct2 in dct.items():
                if col not in colsData: 
                    print(f"Column: {col}    Desciption: {dct2['desc']}")
                    notindict+=1
            if notindict == 0:
                print("NONE .... ALL COLUMNS in DATA FOUND IN SOURCE_TRANFORM DICTIONARY.... WHOO HOOOO") 
            print("\n-----------------")
            
        print("Columns in DATA not in SOURCE-TRANSFORM Dictionary")
        
        for col in colsData:  
            if col not in dct:
                print(col)
                notindata+=1
        if notindata == 0:
                print("NONE .... ALL COLUMNS in SOURCE_TRANFORM DICTIONARY  FOUND IN DATA.... WHOO HOOOO") 
    # layout = [[sg.Button("Quits")],
    #     [sg.Text("Input File "),sg.FilesBrowse(button_text="Get File",target="-GINFO-",initial_folder="/home/joe/",font="CENTAUR 15",file_types=[("All Files","*"),("CSV Files","*.csv"),("TSV Files","*.tsv"),("Excel Files","*.xlsx")],enable_events=True,key='-INPUTFILE-')],
    #     [sg.Input("",key="-CINFO-",enable_events=True)]           
    # ]
    # window = sg.Window('Get File', layout,finalize=True,resizable=True,background_color="#cc4400")
 

# print("DONE")
# ds = "Registration of Charities, Paid Solicitors, Professional Fundraising Consultants, and for-profit Public Benefit Corporations in Colorado"

# cols,desc,w4x4 = getTransCols(ds,datasets)

# fin=open("/home/joe/bic_etl/cdos/business/nonprofit/defs/reg_finan_37wu-kn3g_src_trns_xrefs.json")
# info=json.load(fin)

# compare2Source(info,"/home/joe/bic_etl/cdos/business/nonprofit/data_source/reg_finan.tsv")

## GUI 

In [6]:
def GUI():
    global cim4x4,values
    layout = [
         [sg.Button('Close',font='Courier 15 bold ')],
         [sg.Text("1. Choose Dataset "),sg.Combo(sorted(groupMenu), size=(50,10) , enable_events=True,key='Group Menu',font='Courier 15 bold')],
         [sg.Text("Dataset Info: "),sg.Text("",key="-DINFO-")],
         [sg.Text("2a. Get Source Definition File "),sg.FilesBrowse(button_text="Source Def File",target="-SFILE-",initial_folder="/home/joe/bic_etl/cdos/business/nonprofit/defs",font="CENTAUR 15",file_types=[("All Files","*"),("CSV Files","*.csv"),("TSV Files","*.tsv"),("Excel Files","*.xlsx")],enable_events=True,key='-SOURCEFILE-')],
         [sg.Text("2b. Get Source Data File "),sg.FilesBrowse(button_text="Source Data File",target="-SFILE-",initial_folder="/home/joe/bic_etl/cdos/business/",font="CENTAUR 15",file_types=[("All Files","*"),("CSV Files","*.csv"),("TSV Files","*.tsv"),("Excel Files","*.xlsx")],enable_events=True,key='-SOURCEDATFILE-')],
         [sg.Radio('Use Source Definition File', group_id=1,key="defFIle", default=True), sg.Radio('No Source Definition File', group_id=1,key="dataFile")],
         [sg.Input("",visible=False,key="-SFILE-",enable_events=True)],
         [sg.Text("3. Create Source-Transform Dictionary"),sg.Button('Create',font='Courier 15 bold ')],
         [sg.Text("Input File "),sg.FilesBrowse(button_text="Get File",target="-GINFO-",initial_folder="/home/joe/work",font="CENTAUR 15",file_types=[("All Files","*"),("CSV Files","*.csv"),("TSV Files","*.tsv"),("Excel Files","*.xlsx")],enable_events=True,key='-INPUTFILE-')],
         [sg.Input("",key="-GINFO-",enable_events=True)],
         [sg.Button('Show Field Xrefs')]     
    ]
    window2 = sg.Window('GUI', layout,finalize=True,resizable=True,background_color="#cc4400")
 #   ds = "Registration of Charities, Paid Solicitors, Professional Fundraising Consultants, and for-profit Public Benefit Corporations in Colorado"
    # colsTrans,desc,cim4x4=getTransCols(ds,cimDatasets)
    # coolTrDefs = list(zip(colsTrans,desc))
    # sourceDefFile = '/home/joe/bic_etl/cdos/business/nonprofit/defs/reg_finan_37wu-kn3g_src_fld_defs.json'
    # srcDir = "/home/joe/bic_etl/cdos/nonprofit"
    # ext="reg_finan"
    # cim4x4 = "37wu-kn3g"
    # trFile = lookupInfo[ds][1]
    # srcFile = lookupInfo[ds][0]
    # print("TRFILE ",trFile,srcFile)
    sourceDefFile=""  #  this is needed
    vals=[]
    try: 
        while True:
            wid, event, values = sg.read_all_windows()
            print("EV ",event)
            print(values)
            if event == sg.WIN_CLOSED or event == 'Close':
                window2.close()
                break
            elif event == 'Quit':
                wid.close()
            elif event == "Group Menu":
                print("IN GROUP MENU")
                ds = values["Group Menu"]
                if ds in lookupInfo: 
                   trFile = lookupInfo[ds][1]
                else:
                   print("Datasets is not included in the lookupInfo dictionary...Please add")
                   exit(0)
          #      print("TR FILE ",trFile)
         #       srcFile = lookupInfo[ds][0]
                print("DS ",ds)  
                colsTrans,desc,cim4x4=getTransCols(ds,cimDatasets)
      #          coolTrDefs = sorted(list(zip(colsTrans,desc)))
                ext = getExtention(ds)
                datasetCharId=ext
                string = f"4x4:  {cim4x4}   Extension: {ext}"
                window2["-DINFO-"].update(string)
                print("Extention: ",ext)
            elif event == "Create":
            #    targDir = os.path.join(bicHome, directory,definitionDir)
                targDir="."
                print("VALUES ",values)
             
                if len(sourceDefFile) < 4:
                  #  sourceDefFile = os.path.join(targDir,f"{datasetCharId}_{cim4x4}_src_fld_defs.json") 
                    if len(values["-SOURCEFILE-"]) > 0:
                        sourceDefFile = values["-SOURCEFILE-"]
                        sourceDatFile=""
                    elif len(values["-SOURCEDATFILE-"]):
                        sourceDatFile = values["-SOURCEDATFILE-"]
                        sourceDefFile=""
                 #       colsTrans = {colsTrans[nn]:coolTrDefs[nn]   for nn in range(len(colsTrans))}
                        colsTrans = {colsTrans[nn]:desc[nn]   for nn in range(len(colsTrans))}
                        
               
                    
                print("SDF ",sourceDefFile)
                if sourceDefFile:
                  transTmp = createSrcTransFieldXref(cim4x4,colsTrans,sourceDefFile,targDir)
                  showTransDesc(ds,coolTrDefs)
                elif  sourceDatFile:
                  origTmp = createSrcTransFieldXrefReverse(cim4x4,colsTrans,sourceDatFile,targDir)
                  print("OTMP ",origTmp)
                  showSrcCols(ds,origTmp)
                
                if len(trFile) < 25: 
                   showFile(f"/home/joe/bic_etl/cdos/business/nonprofit/scripts/{trFile}")
                else:
                   showFile(f"{trFile}")
                    
            elif event == "Write XREF File":
                try:
                  #  targDir = os.path.join(bicHome, directory,definitionDir)
                 #   createDir(targDir)
                    targDir="."
                    print("Goint to Write")
                    vals=values
                    if sourceDefFile:
                       writeFieldFileN2(cim4x4,ds,values,targDir,ext)
                    elif sourceDatFile:
                       writeFieldFileN3(cim4x4,ds,values,targDir,ext)
                        
                except Exception as err:
                    print("ERROR ",err)

            elif event == "Show Field Xrefs":
               # file=values['-GINFO-']
                # targDir = os.path.join(bicHome, directory,definitionDir)
                # file = os.path.join(targDir,f"{file}")
                try:
                    showSrc2TransXrefs(inFile)
                except Exception as err:
                    print("NOPE NOPE ",err)
                
            elif event == "-GINFO-":
                inFile = values["-GINFO-"]
              #  dfSource = getFile("Local",file,window2)

            elif event == "Cmpr to Source Data":
                info=wid.metadata
                compare2Source()
            
    except Exception as err:
        print("ERROR ERROR ")
        print(err)
    return vals,colsTrans,desc,coolTrDefs

print("STARTING")
vals,colsTrans,desc,coolTrDefs = GUI()

print("DONE")


STARTING
EV  Group Menu
{'Group Menu': 'Master List in Colorado', '-SOURCEFILE-': '', '-SOURCEDATFILE-': '', 'defFIle': True, 'dataFile': False, '-SFILE-': '', '-INPUTFILE-': '', '-GINFO-': ''}
IN GROUP MENU
DS  Master List in Colorado
Extention:  
EV  -SFILE-
{'Group Menu': 'Master List in Colorado', '-SOURCEFILE-': '', '-SOURCEDATFILE-': '/home/joe/bic_etl/cdos/business/business/data_source/masterlist.tsv', 'defFIle': True, 'dataFile': False, '-SFILE-': '/home/joe/bic_etl/cdos/business/business/data_source/masterlist.tsv', '-INPUTFILE-': '', '-GINFO-': ''}
EV  Create
{'Group Menu': 'Master List in Colorado', '-SOURCEFILE-': '', '-SOURCEDATFILE-': '/home/joe/bic_etl/cdos/business/business/data_source/masterlist.tsv', 'defFIle': True, 'dataFile': False, '-SFILE-': '/home/joe/bic_etl/cdos/business/business/data_source/masterlist.tsv', '-INPUTFILE-': '', '-GINFO-': ''}
VALUES  {'Group Menu': 'Master List in Colorado', '-SOURCEFILE-': '', '-SOURCEDATFILE-': '/home/joe/bic_etl/cdos/busines

NameError: name 'coolTrDefs' is not defined

In [9]:
a={'SOURCE:recordCountyFilingId:0': 'Record Id #', 'LISTBOX:recordCountyFilingId:0': 'recordCountyFilingId', 'DESC:recordCountyFilingId:0': 'Filing id associated with the county where the record was filed', 'NOTES:recordCountyFilingId:0': '', 'DATE:recordCountyFilingId:0': '2025-1-6', 'DUPE:recordCountyFilingId:0': True, 'SOURCE:recordIdDate:1': 'Record Id #', 'LISTBOX:recordIdDate:1': 'recordIdDate', 'DESC:recordIdDate:1': 'Date record was filed', 'NOTES:recordIdDate:1': '', 'DATE:recordIdDate:1': '2025-1-6', 'DUPE:recordIdDate:1': True, 'SOURCE:recordCountyFiling:2': 'Record Id #', 'LISTBOX:recordCountyFiling:2': 'recordCountyFiling', 'DESC:recordCountyFiling:2': 'County where record was filed', 'NOTES:recordCountyFiling:2': '', 'DATE:recordCountyFiling:2': '2025-1-6', 'DUPE:recordCountyFiling:2': True, 'SOURCE:assignee:3': 'Assignee(s)', 'LISTBOX:assignee:3': 'assignee', 'DESC:assignee:3': 'Assignee listed on the effective financing statement', 'NOTES:assignee:3': '', 'DATE:assignee:3': '2025-1-6', 'DUPE:assignee:3': False, 'SOURCE:cropYear:4': 'Crop Year(s)', 'LISTBOX:cropYear:4': 'cropYear', 'DESC:cropYear:4': 'Years that are covered by the financing statement for the farm product. Listed as ALL if it covers all years', 'NOTES:cropYear:4': '', 'DATE:cropYear:4': '2025-1-6', 'DUPE:cropYear:4': False, 'SOURCE:counties:5': 'County(ies)', 'LISTBOX:counties:5': 'counties', 'DESC:counties:5': 'County or counties where the effective financing statement is valid', 'NOTES:counties:5': '', 'DATE:counties:5': '2025-1-6', 'DUPE:counties:5': False, 'SOURCE:recordIdVerbatim:6': 'Record Id #', 'LISTBOX:recordIdVerbatim:6': 'recordIdVerbatim', 'DESC:recordIdVerbatim:6': 'Full record id field', 'NOTES:recordIdVerbatim:6': '', 'DATE:recordIdVerbatim:6': '2025-1-6', 'DUPE:recordIdVerbatim:6': False, 'SOURCE:additionalDebtors:7': 'Additional Debtors', 'LISTBOX:additionalDebtors:7': 'additionalDebtors', 'DESC:additionalDebtors:7': 'Additional debtors, if applicable, listed on the effective financing statement', 'NOTES:additionalDebtors:7': '', 'DATE:additionalDebtors:7': '2025-1-6', 'DUPE:additionalDebtors:7': False, 'SOURCE:amendmentIdVerbatim:8': 'Amendment ID #(s)', 'LISTBOX:amendmentIdVerbatim:8': 'amendmentIdVerbatim', 'DESC:amendmentIdVerbatim:8': 'Full amendment id field', 'NOTES:amendmentIdVerbatim:8': '', 'DATE:amendmentIdVerbatim:8': '2025-1-6', 'DUPE:amendmentIdVerbatim:8': False, 'SOURCE:recordId:9': 'Record Id #', 'LISTBOX:recordId:9': 'recordId', 'DESC:recordId:9': 'Unique id for record', 'NOTES:recordId:9': '', 'DATE:recordId:9': '2025-1-6', 'DUPE:recordId:9': True, 'SOURCE:securedParty:10': 'Secured Party(ies)', 'LISTBOX:securedParty:10': 'securedParty', 'DESC:securedParty:10': 'Secured party listed on the effective financing statement', 'NOTES:securedParty:10': '', 'DATE:securedParty:10': '2025-1-6', 'DUPE:securedParty:10': False, 'SOURCE:additionalDebtorId:11': 'Additional Debtors', 'LISTBOX:additionalDebtorId:11': 'additionalDebtorId', 'DESC:additionalDebtorId:11': 'FEIN associated with additional debtor, if applicable', 'NOTES:additionalDebtorId:11': '', 'DATE:additionalDebtorId:11': '2025-1-6', 'DUPE:additionalDebtorId:11': True, 'SOURCE:debtorAddress:12': 'Debtor Address', 'LISTBOX:debtorAddress:12': 'debtorAddress', 'DESC:debtorAddress:12': 'Address of the debtor on the effective financing statement', 'NOTES:debtorAddress:12': '', 'DATE:debtorAddress:12': '2025-1-6', 'DUPE:debtorAddress:12': False, 'SOURCE:debtorId:13': 'Debtor ID #', 'LISTBOX:debtorId:13': 'debtorId', 'DESC:debtorId:13': 'FEIN associated with the debtor for the effective financing statement', 'NOTES:debtorId:13': '', 'DATE:debtorId:13': '2025-1-6', 'DUPE:debtorId:13': False, 'SOURCE:debtorName:14': 'Debtor Name', 'LISTBOX:debtorName:14': 'debtorName', 'DESC:debtorName:14': 'Name of the individual debtor or debtor organization for the effective financing statement', 'NOTES:debtorName:14': '', 'DATE:debtorName:14': '2025-1-6', 'DUPE:debtorName:14': False, 'SOURCE:amendmentId:15': 'Amendment ID #(s)', 'LISTBOX:amendmentId:15': 'amendmentId', 'DESC:amendmentId:15': 'Full amendment id fields', 'NOTES:amendmentId:15': '', 'DATE:amendmentId:15': '2025-1-6', 'DUPE:amendmentId:15': True, 'SOURCE:additionalFarmProduct:16': 'Additional Farm Product(s)', 'LISTBOX:additionalFarmProduct:16': 'additionalFarmProduct', 'DESC:additionalFarmProduct:16': 'Additional farm products associated with the effective financing statement', 'NOTES:additionalFarmProduct:16': '', 'DATE:additionalFarmProduct:16': '2025-1-6', 'DUPE:additionalFarmProduct:16': False, 'SOURCE:farmProduct:17': 'Farm Product', 'LISTBOX:farmProduct:17': 'farmProduct', 'DESC:farmProduct:17': 'Farm product associated with effective financing statement', 'NOTES:farmProduct:17': '', 'DATE:farmProduct:17': '2025-1-6', 'DUPE:farmProduct:17': False, 'SOURCEC:CUST:18': '', 'TRANSC:CUST:18': '', 'DESCC:CUST:18': '', 'NOTESC:CUST:18': '', 'DATEC:CUST:18': '2025-1-6', 'DUPEC:CUST:18': False, 'SOURCEC:CUST:19': '', 'TRANSC:CUST:19': '', 'DESCC:CUST:19': '', 'NOTESC:CUST:19': '', 'DATEC:CUST:19': '2025-1-6', 'DUPEC:CUST:19': False, 'SOURCEC:CUST:20': '', 'TRANSC:CUST:20': '', 'DESCC:CUST:20': '', 'NOTESC:CUST:20': '', 'DATEC:CUST:20': '2025-1-6', 'DUPEC:CUST:20': False, 'SOURCEC:CUST:21': '', 'TRANSC:CUST:21': '', 'DESCC:CUST:21': '', 'NOTESC:CUST:21': '', 'DATEC:CUST:21': '2025-1-6', 'DUPEC:CUST:21': False, 'SOURCEC:CUST:22': '', 'TRANSC:CUST:22': '', 'DESCC:CUST:22': '', 'NOTESC:CUST:22': '', 'DATEC:CUST:22': '2025-1-6', 'DUPEC:CUST:22': False, 'SOURCEC:CUST:23': '', 'TRANSC:CUST:23': '', 'DESCC:CUST:23': '', 'NOTESC:CUST:23': '', 'DATEC:CUST:23': '2025-1-6', 'DUPEC:CUST:23': False, 'SOURCEC:CUST:24': '', 'TRANSC:CUST:24': '', 'DESCC:CUST:24': '', 'NOTESC:CUST:24': '', 'DATEC:CUST:24': '2025-1-6', 'DUPEC:CUST:24': False, 'SOURCEC:CUST:25': '', 'TRANSC:CUST:25': '', 'DESCC:CUST:25': '', 'NOTESC:CUST:25': '', 'DATEC:CUST:25': '2025-1-6', 'DUPEC:CUST:25': False, 'SOURCEC:CUST:26': '', 'TRANSC:CUST:26': '', 'DESCC:CUST:26': '', 'NOTESC:CUST:26': '', 'DATEC:CUST:26': '2025-1-6', 'DUPEC:CUST:26': False, 'SOURCEC:CUST:27': '', 'TRANSC:CUST:27': '', 'DESCC:CUST:27': '', 'NOTESC:CUST:27': '', 'DATEC:CUST:27': '2025-1-6', 'DUPEC:CUST:27': False, 'SOURCEC:CUST:28': '', 'TRANSC:CUST:28': '', 'DESCC:CUST:28': '', 'NOTESC:CUST:28': '', 'DATEC:CUST:28': '2025-1-6', 'DUPEC:CUST:28': False, 'SOURCEC:CUST:29': '', 'TRANSC:CUST:29': '', 'DESCC:CUST:29': '', 'NOTESC:CUST:29': '', 'DATEC:CUST:29': '2025-1-6', 'DUPEC:CUST:29': False, 'SOURCEC:CUST:30': '', 'TRANSC:CUST:30': '', 'DESCC:CUST:30': '', 'NOTESC:CUST:30': '', 'DATEC:CUST:30': '2025-1-6', 'DUPEC:CUST:30': False, 'SOURCEC:CUST:31': '', 'TRANSC:CUST:31': '', 'DESCC:CUST:31': '', 'NOTESC:CUST:31': '', 'DATEC:CUST:31': '2025-1-6', 'DUPEC:CUST:31': False}

In [ ]:
a

In [10]:
hist={}
for key,val in a.items():
    spl=key.split(":")
    nn=spl[2]
    typ=spl[0]
    trCol=spl[1]
    if nn not in hist:
        hist[nn]={}
    # if trCol not in hist[nn]:
    #     hist[nn][trCol]={}
        
    hist[nn][typ]=val

In [21]:
dupeCols={}
mapped={}
w4x4="1234-abcd"
mapped[w4x4]={}
for nn in hist:
    if 'SOURCE' in hist[nn]:
        indx = hist[nn]['SOURCE']
        xref = hist[nn]['LISTBOX']
        desc = hist[nn]['DESC']
        date = hist[nn]['DATE']
        
        if 'DUPE' in hist[nn]:
            dupe = hist[nn]['DUPE']
        else:
            dupe = False
    
        if dupe:
            if indx not in dupeCols:
                dupeCols[indx]=0
            dupeCols[indx]+=1
            num=dupeCols[indx]
            indx+=f"_{num}"
        mapped[w4x4][indx]={}
        mapped[w4x4][indx]['xref'] = xref 
        mapped[w4x4][indx]['desc'] = desc
        mapped[w4x4][indx]['date'] = date
        mapped[w4x4][indx]['dupe'] = dupe
        if dupe:
            mapped[w4x4][indx]['srcflag']=1
            
        
    elif 'SOURCEC' in hist[nn]: 
        indx = hist[nn]['SOURCEC']
        if len(indx) > 0:
            xref = hist[nn]['TRANSC']
            desc = hist[nn]['DESCC']
            date = hist[nn]['DATEC']
            if 'DUPEC' in hist[nn]:
                dupe = hist[nn]['DUPEC']
            else:
                dupe = False
            if dupe:
                if indx not in dupeCols:
                    dupeCols[indx]=0
                dupeCols[indx]+=1
                num=dupeCols[indx]
                indx+=f"_{num}"

            mapped[w4x4][indx]={}
            mapped[w4x4][indx]['xref'] = xref 
            mapped[w4x4][indx]['desc'] = desc
            mapped[w4x4][indx]['date'] = date
            mapped[w4x4][indx]['dupe'] = dupe
            if dupe:
                mapped[w4x4][indx]['srcflag']=1
            print(f"CC {len(indx)}  {indx}  {xref}  {desc}  {date}  {dupe}")


In [22]:
mapped

{'1234-abcd': {'Record Id #_1': {'xref': 'recordCountyFilingId',
   'desc': 'Filing id associated with the county where the record was filed',
   'date': '2025-1-6',
   'dupe': True,
   'srcflag': 1},
  'Record Id #_2': {'xref': 'recordIdDate',
   'desc': 'Date record was filed',
   'date': '2025-1-6',
   'dupe': True,
   'srcflag': 1},
  'Record Id #_3': {'xref': 'recordCountyFiling',
   'desc': 'County where record was filed',
   'date': '2025-1-6',
   'dupe': True,
   'srcflag': 1},
  'Assignee(s)': {'xref': 'assignee',
   'desc': 'Assignee listed on the effective financing statement',
   'date': '2025-1-6',
   'dupe': False},
  'Crop Year(s)': {'xref': 'cropYear',
   'desc': 'Years that are covered by the financing statement for the farm product. Listed as ALL if it covers all years',
   'date': '2025-1-6',
   'dupe': False},
  'County(ies)': {'xref': 'counties',
   'desc': 'County or counties where the effective financing statement is valid',
   'date': '2025-1-6',
   'dupe': Fals

In [ ]:
hist

In [ ]:
from datetime import date
from dateutil.relativedelta import relativedelta

today = date.today()
three_months_ago = today - relativedelta(months=3)

print(three_months_ago.day)
day=1
print(date(three_months_ago.year,three_months_ago.month,1))

In [ ]:
def getInfo(dataset):
    name=dataset["resource"]["name"]
    start=dataset["resource"]["createdAt"]
    dif=datetime.today()-datetime.strptime(dataset["resource"]["createdAt"],"%Y-%m-%dT%H:%M:%S.%fZ")
    day=dif.days
    # for key,val in dataset["resource"]["page_views"].items():
    #     print(key,val)
    view=dataset["resource"]["page_views"]["page_views_total"]
    viewRate=int(dataset["resource"]["page_views"]["page_views_total"])/dif.days
    
    down=dataset["resource"]["download_count"]
    downRate=int(dataset["resource"]["download_count"])/dif.days
    return name,start,day,view,viewRate,down,downRate


datasets={}
for dataset in cimDatasets:
    name=dataset["resource"]["name"]
    id=dataset["resource"]["id"]
    datasets[id]=dataset
    name,start,day,view,viewRate,down,downRate = getInfo(dataset)
       



hist={}
names=[]
starts=[]
difs=[]
days=[]
views=[]
vRs=[]
downs=[]
dRs=[]

Pnames=[]
Pstarts=[]
Pdifs=[]
Pdays=[]
Pviews=[]
PvRs=[]
Pdowns=[]
PdRs=[]
#getType="filter"
#getType="map"
getType="dataset"

for dataset in cimDatasets:
    tp=dataset["resource"]["type"]
    if tp not in hist:
        hist[tp]=0
    hist[tp]+=1
    P=False
    if tp == getType:
        name=dataset["resource"]["name"]
        print(name,dataset["resource"]["id"],dataset["resource"]["parent_fxf"])
        start=dataset["resource"]["createdAt"]
        if len(dataset["resource"]["parent_fxf"]) > 0:
            idP=dataset["resource"]["parent_fxf"][0]
            P=True
        dif=datetime.today()-datetime.strptime(dataset["resource"]["createdAt"],"%Y-%m-%dT%H:%M:%S.%fZ")
        day=dif.days
        # for key,val in dataset["resource"]["page_views"].items():
        #     print(key,val)
        view=dataset["resource"]["page_views"]["page_views_total"]
        viewRate=int(dataset["resource"]["page_views"]["page_views_total"])/dif.days
        
        down=dataset["resource"]["download_count"]
        downRate=int(dataset["resource"]["download_count"])/dif.days
        names.append(name)
        starts.append(start)
        days.append(day)
        views.append(view)
        vRs.append(viewRate)
        downs.append(down)
        dRs.append(downRate)
        datasetP=datasets[idP]
        if P:
            nameP,startP,dayP,viewP,viewRateP,downP,downRateP = getInfo(datasetP)
        else:
            nameP,startP,dayP,viewP,viewRateP,downP,downRateP="","","","","","",""
        Pnames.append(nameP)
        Pstarts.append(startP)
        Pdays.append(dayP)
        Pviews.append(viewP)
        PvRs.append(viewRateP)
        Pdowns.append(downP)
        PdRs.append(downRateP)
        
df=pd.DataFrame({
    "Name":names,
    "Started":starts,
    "Days Existed":days,
    "Total Views":views,
    "View Rate (/day)":vRs,
    "Parent Total Views":Pviews,
    "Parent View Rate (/day)":PvRs,
    "Total Downloads":downs,
    "Downloads Rate (/day)":dRs,
    "Parent Total Downloads":Pdowns,
    "Parent Downloads Rate (/day)":PdRs,
    "Parent Name":Pnames,
    "Parent Started":Pstarts,
    "Parent Days Existed":Pdays
    
   
})

In [ ]:
cim_url_query = 'data.colorado.gov'
with Socrata(cim_url_query, None) as client:
   metadata = client.get_metadata("ej2c-jkvh")

xrefs={}
for col in metadata["columns"]:
    print(col["name"],col["description"])
    name=col["name"].strip()
    desc=col["description"].strip()
    xrefs[name]=desc
    

In [ ]:
xrefs

In [ ]:
import requests

In [ ]:
url="https://data.colorado.gov/api/views/ej2c-jkvh.json"
response = requests.get(url, stream=True)
response.raise_for_status()



In [ ]:
response.json()

## CDOS Business Datasets

The GUI above works for when we have a source dictionary file that lists the original source names in the data and their definitions.  John Coniff provided a word document of this for the Charity Datasets to me in  2023, and I created individual json files for each dataset that lists the field name and its description.  This does not exist for the CDOS Business datasets, so I reverse engineered a library from CIM using the field names and descriptions, then reverse engineered a source-tranform cross-reference dictionary, using the codes below. 

## Business Entities

In [ ]:
dataset : ""
for ds in cimDatasets:
   if ds['resource']['name'] == "Paid Solicitor Solicitation Notices in Colorado":
    dataset = ds
    break

In [ ]:
w4x4 = dataset['resource']['id']
desc = dataset['resource']['columns_description']
cols = dataset['resource']['columns_name']
flds = dataset['resource']['columns_field_name']

In [ ]:
defs = {}
for nn,col in enumerate(cols):
    defs[col]=desc[nn]
    if col != flds[nn]:
   #     print(col,flds[nn])
        print(f'"{col}": ""')
        
        

In [ ]:
xrefs = {
"farmProduct": "Farm Product"
"debtorName": "Debtor Name"
"debtorId": "Debtor ID #"
"cropYear": "Crop Year(s)"
"debtorAddress": "Debtor Address"
"additionalDebtors": "Additional Debtors"
"additionalDebtorId": ""
"securedParty": "Secured Party(ies)"
"additionalFarmProduct": "Additional Farm Product(s)"
"recordIdVerbatim": ""
"recordId": "Record Id #"
"recordIdDate": ""
"assignee": "Assignee(s)"
"recordCountyFilingId": ""
"recordCountyFiling": "County(ies)"
"amendmentIdVerbatim": ""
"amendmentId": "Amendment ID #(s)"
        }

xrefsSrc2Trans = {src:trns for trns,src in xrefs.items()}

In [ ]:
xrefsSrc2Trans.items()

In [ ]:
xrefDefs = {}
xrefDefs[w4x4]={}

print(xrefDefs)
for src,trans in xrefsSrc2Trans.items():
    print(trans,today)
    xrefDefs[w4x4][src]={
        "xref":trans,
        "desc":defs[trans.lower()],
        "notes":"",
        "date":today
    }

In [ ]:
ext="bus_ent"
outfile=f"{ext}_{w4x4}_src_trns_xrefs.json"
with open(outfile,"w") as jfile:
            jfile.write(json.dumps(xrefDefs))

## Trade Marks

In [ ]:
dataset : ""
for ds in cimDatasets:
   if ds['resource']['name'] == "Trademarks for Businesses in Colorado":
    dataset = ds
    break

In [ ]:
w4x4 = dataset['resource']['id']
desc = dataset['resource']['columns_description']
cols = dataset['resource']['columns_name']
flds = dataset['resource']['columns_field_name']

In [ ]:
defs = {}
for nn,col in enumerate(cols):
    defs[col]=desc[nn]
    if col.lower() != flds[nn]:
        print(col,flds[nn])
        

In [ ]:
defs

In [ ]:
xrefs = {
"entityId"           : "Entity Id",
"masterTrademarkId"  : "Master Trademark ID",
"entityId"           : "Entity ID",
"entityName"         : "Registrant Name",
"status"             : "Trademark Status",
"type"               : "Trademark Type",
"registrationDate"   : "Registration Date",
"dateFirstUsed"      : "Date of First Use",
"expirationDate"     : "Expiration Date",
"goodServiceClass"   : "Goods and Services Class",
"goodServiceDetail"  : "Goods and Services Detail",
"description"        : "Trademark Description",
"trademarkForm"      : "Trademark Form",
"entityStatus"       : "Entity Status",
"entityFormDate"     :  "Entity Form Date" 
    
}

xrefsSrc2Trans = {src:trns for trns,src in xrefs.items()}

In [ ]:
xrefDefs = {}
xrefDefs[w4x4]={}

print(xrefDefs)
for src,trans in xrefsSrc2Trans.items():
    print(trans,today)
    xrefDefs[w4x4][src]={
        "xref":trans,
        "desc":defs[trans],
        "notes":"",
        "date":today
    }

In [ ]:
ext="tra_mrks"
outfile=f"{ext}_{w4x4}_src_trns_xrefs.json"
with open(outfile,"w") as jfile:
            jfile.write(json.dumps(xrefDefs))

## Trade Names

In [ ]:
dataset : ""
for ds in cimDatasets:
   if ds['resource']['name'] == "Trade Names for Businesses in Colorado":
    dataset = ds
    break

In [ ]:
w4x4 = dataset['resource']['id']
desc = dataset['resource']['columns_description']
cols = dataset['resource']['columns_name']
flds = dataset['resource']['columns_field_name']

In [ ]:
defs = {}
for nn,col in enumerate(cols):
    defs[col]=desc[nn]
    if col.lower() != flds[nn]:
        print(col,flds[nn])
        

In [ ]:
xrefs = {"entityStatus"    :"Entity Status",
"entityFormDate"          :"Entity Form Date",
"masterTradenameId"      : "Mstr Trdnm Id",
"tradenameDescription"   :"Trdnm Dscr",
"tradenameForm"          :"Tradename Form",
"effectiveDate"          :"Add Dtm",
"firstName"              :"First Nm",
"middleName"             :"Middle Nm",
"lastName"               :"Last Nm",
"suffix"                 :"Suffix",
"registrantOrganization" :"Registrant Organization",
"address1"               :"Address1",
"address2"               :"Address2",
"city"                   :"City",
"state"                  :"State",
"zipCode"                :"Zip",
"country"                :"Country",
"mailingAddress1"        :"Mailing Address 1",
"mailingAddress2"        :"Mailing Address 2",
"mailingCity"            : "Mailing City",
"mailingState"           :"Mailing State",
"mailingZipCode"         :"Mailing Zip",
"mailingCountry"         :"Mailing Country",
"dateAdded"              : "Add Dtm",
"Entity ID"               :"Entity ID",
"effectiveDate"          : "Effective Dtm"
       }

# xrefsSrc2Trans = {src:trns for trns,src in xrefs.items() if src in xrefsSrc2Trans exit(1)}
xrefsSrc2Trans={}
chkTrns={}
for trns,src in xrefs.items():
    if src not in xrefsSrc2Trans:
        xrefsSrc2Trans[src]=trns
    else:
        print("DUplicate Source Columns: ",src)
        print("Stopping Short")
        break
    
    if trns in chkTrns:
        print("Duplicate TRANSFORM Column: ",trns)
    chkTrns[trns]=1
    


In [ ]:
## Create the Dictionary Lookup 
xrefDefs = {}
xrefDefs[w4x4]={}

print(xrefDefs)
for src,trans in xrefsSrc2Trans.items():
    print(trans,today)
    xrefDefs[w4x4][src]={
        "xref":trans,
        "desc":defs[trans],
        "notes":"",
        "date":today
    }

### Check CIM against New Library/Dictionary

In [ ]:
cim = set(defs.keys())
xen = set(xrefs.keys())

print("Columns on CIM NOT in Our Dictionary Lookups")
print(cim-xen)

print("\nColumns in OUR Dictionary Lookup NOT on CIM")
print(xen-cim)

In [ ]:
df = pd.read_csv("https://data.colorado.gov/resource/u7sb-g482.csv")
#df = pd.read_csv("/home/joe/bic_etl/cdos/business/business/data_source/tradenames.tsv",encoding="latin",delimiter="\t")

for col in sorted(df.columns):
   print(col)

In [ ]:
df['entityid'].value_counts()

In [ ]:
## Write Output Dictionary
ext="tra_nams"
outfile=f"{ext}_{w4x4}_src_trns_xrefs.json"
with open(outfile,"w") as jfile:
            jfile.write(json.dumps(xrefDefs))
            print(outfile," written")

In [ ]:
flds

In [ ]:
fin = open("/home/joe/bic_etl/cdos/business/business/data_source/tradenames.tsv",encoding="latin")

head=fin.readline()
header=head.split("\t")
print(len(header))
hist={}
for line in fin:
    spl=line.split("\t")
    nn=len(spl)
    
    if nn in hist:
        hist[nn]+=1
    else:
        hist[nn]=1
    

In [ ]:
hist

## Master List

In [ ]:
dataset : ""
for ds in cimDatasets:
   if ds['resource']['name'] == "Master List in Colorado":
    dataset = ds
    break

In [ ]:
w4x4 = dataset['resource']['id']
desc = dataset['resource']['columns_description']
cols = dataset['resource']['columns_name']
flds = dataset['resource']['columns_field_name']

In [ ]:
defs = {}
for nn,col in enumerate(cols):
    defs[col]=desc[nn]
    if col.lower() != flds[nn]:
        print(col,flds[nn])
        

In [ ]:
print(len(cols))
for col in sorted(cols):
    print(f"'{col}': '',")

In [ ]:
xrefs = {
'additionalDebtorId': 'additionalDebtorId',
'additionalDebtors': 'Additional Debtors',
'additionalFarmProduct': 'Additional Farm Product(s)',
'amendmentId': 'Amendment ID #(s)',
'amendmentIdVerbatim': 'amendmentIdVerbatim',
'assignee': 'Assignee(s)',
'counties': 'County(ies)',
'cropYear': 'Crop Year(s)',
'debtorAddress': 'Debtor Address',
'debtorId': 'Debtor ID #',
'debtorName': 'Debtor Name',
'farmProduct': 'Farm Product',
'recordCountyFiling': 'recordCountyFiling',
'recordCountyFilingId': 'recordCountyFilingId',
'recordId': 'Record Id #',
'recordIdDate': 'recordIdDate',
'recordIdVerbatim': 'recordIdVerbatim',
'securedParty': 'Secured Party(ies)'}




xrefsSrc2Trans={}
chkTrns={}
for trns,src in xrefs.items():
    if src not in xrefsSrc2Trans:
        xrefsSrc2Trans[src]=trns
    else:
        print("DUplicate Source Columns: ",src)
   #     print("Stopping Short")
    #    break
    
    if trns in chkTrns:
        print("Duplicate TRANSFORM Column: ",trns)
    chkTrns[trns]=1
    


In [ ]:
## Create the Dictionary Lookup 
xrefDefs = {}
xrefDefs[w4x4]={}

for src,trans in xrefsSrc2Trans.items():
    xrefDefs[w4x4][src]={
        "xref":trans,
        "desc":defs[trans],
        "notes":"",
        "date":today,
        "source":"cdos"
    }

In [ ]:
calcs = [
'recordIdVerbatim',
'recordIdDate',
'recordCountyFilingId',
'recordCountyFiling',
'amendmentIdVerbatim',
'additionalDebtorId']

In [ ]:
# FOr the fields xentity calculates, set their source to calculated 
for col in calcs:
    xrefDefs[w4x4][col]['source'] = 'calculated'
    xrefDefs[w4x4][col]['notes'] = 'This field is calculated or created by Xentity'
    
    print(xrefDefs[w4x4][col]['source'])

In [ ]:
xrefDefs

In [ ]:
## Write Output Dictionary
ext="mst_lst"
outfile=f"{ext}_{w4x4}_src_trns_xrefs.json"
with open(outfile,"w") as jfile:
            jfile.write(json.dumps(xrefDefs))
            print(outfile," written")

In [ ]:
dfC = pd.read_csv("https://data.colorado.gov/resource/ej2c-jkvh.json?$limit=10000000")
dfC.columns

In [ ]:
dfO = pd.read_csv("UCCMstrLB1.txt",delimiter="\t",encoding="latin")

In [ ]:
for col in sorted(dfO.columns):
    print(col)

In [ ]:
dfC = pd.read_csv("https://data.colorado.gov/resource/ej2c-jkvh.csv?$limit=10000000")
dfC.columns

In [ ]:
print(dfO.shape)
print(dfC.shape)

In [ ]:
for col in sorted(dfC.columns):
    print(col)

In [ ]:
for col in sorted(dfO.columns):
    print(col)

In [ ]:
display(dfC.head())

In [ ]:
xrefs

In [ ]:
no=[]
for col,src in xrefs.items():
    if len(src) == 0:
        no.append(col)

In [ ]:
no

In [ ]:
display(dfO.head(2))

In [ ]:
display(dfC.head(2))

In [ ]:
display(dfO.head(2))

In [ ]:
dfO["Description of Farm Product(s)"].value_counts()

In [ ]:
dfO.shape

In [ ]:
dfO.loc[dfO["Debtor Name"] == "COEN, MONTY"]

In [ ]:
dfC.loc[dfC["debtorname"] == "COEN, MONTY"]

## Business Entity Transaction History

In [ ]:
dataset : ""
for ds in cimDatasets:
   if ds['resource']['name'] == "Business Entity Transaction History":
    dataset = ds
    break

In [ ]:
w4x4 = dataset['resource']['id']
desc = dataset['resource']['columns_description']
cols = dataset['resource']['columns_name']
flds = dataset['resource']['columns_field_name']

In [ ]:
defs = {}
for nn,col in enumerate(cols):
    defs[col]=desc[nn]
    if col.lower() != flds[nn]:
        print(col,flds[nn])
        

In [ ]:
## Write Output Dictionary
ext="bus_trns"
outfile=f"{ext}_{w4x4}_src_trns_xrefs.json"
with open(outfile,"w") as jfile:
            jfile.write(json.dumps(xrefDefs))
            print(outfile," written")

In [ ]:
df = pd.read_csv("/home/joe/bic_etl/cdos/business/business/data_source/corphist-2.tsv",encoding="latin",quoting=3,delimiter="\t")

In [ ]:
df.columns

In [ ]:
#df1 = pd.read_csv("/home/joe/bic_etl/cdos/business/business/data_source/corphist-1.tsv",quoting=3,delimiter="\t")
df1 = pd.read_csv("/home/joe/bic_etl/cdos/business/business/data_source/corphist-1.tsv",delimiter="\t")

In [ ]:
for col in df1.columns:
    print(col.strip())
    for c in col:
        print(c,ord(c))

In [ ]:
fin = open("/home/joe/bic_etl/cdos/business/business/data_source/corphist-2.tsv",encoding="latin")
line=fin.readline()
#line=line.replace("\uFEFF","")
line.find("\uFEFF")
# for n,c in enumerate(line):
#     print(n,c,ord(c))
fout = open("corphist2.tsv","w")
nlines=0
for line in fin:
    nlines+=1
    line=line.strip()
    fout.write(f"{line}\n")
    
print("Total Lines: ",nlines)

In [ ]:
fin = open("/home/joe/bic_etl/cdos/business/business/data_source/corphist-1.tsv",encoding="utf-8-sig")
header = fin.readline()
for c in header:
    print(c,ord(c))

In [ ]:
fin = open("/home/joe/bic_etl/cdos/business/business/data_source/corphist-1.tsv",encoding="utf-8-sig")
sig = fin.readlines()
fin.close()

fin2 = open("/home/joe/bic_etl/cdos/business/business/data_source/corphist-1.tsv")
org = fin2.readlines()
fin2.close()

In [ ]:
for nn,lorig in enumerate(org):
    lsig=sig[nn]
    for no,co in enumerate(lorig):
        cl=lsig[no:no+1]
        if co != cl:
            if len(co) > 0 and len(cl) > 0: 
                print(no,co,cl,ord(co),ord(cl))
            else:
                print(no,co,cl)
                
    
        
print("Total Records Checked",nn) 

In [ ]:
def foo(encoding=''):
    print('bar:', type(encoding), encoding)

s = 'encoding = latin'
d = {k.strip(): v.strip() for k, v in [s.split('=', 1)]}
print('d:', type(d), d)
foo(**d)

In [ ]:

s = 'encoding = latin'
if len(s) > 0:
    d = {k.strip(): v.strip() for k, v in [s.split('=', 1)]}
else:
    d=""
print(d)

if len(d) > 0:
    op = open("/home/joe/bic_etl/cdos/business/business/data_source/corphist-2.tsv",**d)
else:    
    op = open("/home/joe/bic_etl/cdos/business/business/data_source/corphist-2.tsv")
#fin = open("/home/joe/bic_etl/cdos/business/business/data_source/corphist-2.tsv",**d)
fin = op

header = fin.readlines()

In [ ]:
fin = open("/home/joe/bic_etl/cdos/business/business/data_transformed/corphist.csv")
orig = fin.readlines()
fin.close()

In [ ]:
fin = open("/home/joe/bic_etl/cdos/business/business/data_transformed/corphist.tsv")
reader2 = csv.reader(fin,delimiter="\t",quoting=3)
new=[]
for row in reader2:
    new.append(row)

In [ ]:
new[0]

In [ ]:
histN={}
for line in new:
    nn = len(line)
    if nn in histN:
        histN[nn]+=1
    else:
        histN[nn]=1
        
print(histN)

In [ ]:
fin = open("/home/joe/bic_etl/cdos/business/business/data_transformed/corphist.csv")
reader1 = csv.reader(fin)
orig=[]
for row in reader1:
    orig.append(row)

In [ ]:
invt= {}
invtdy= {}

for row in orig[1:]:
    spl = row[4].split("/")
    mo = int(spl[0])
    dy = int(spl[1])
    yr = int(spl[2][:4])         
    if yr in invt and mo in invt[yr]:
        invt[yr][mo]+=1
    elif yr in invt and mo not in invt[yr]:
        invt[yr][mo]=1
    elif yr not in invt:
        invt[yr]={}
        invt[yr][mo]=1
        
    if yr not in invtdy:
        invtdy[yr]={}
    if mo not in invtdy[yr]:
        invtdy[yr][mo]={}
    if dy in invtdy[yr][mo]:
        invtdy[yr][mo][dy]+=1
    else:
        invtdy[yr][mo][dy]=1

In [ ]:
print("YEAR   JAN   FEB   MAR   APR   MAY   JUN   JUL   AUG   SEP   OCT   NOV   DEC")
for yr,inv in sorted(invt.items()):
    print(f"{yr:4d} ",end="")
    for mo in range(1,13):
        if mo in inv:
            cnt=inv[mo]
        else:
            cnt=0
        print(f"{cnt:5d} ",end="")
    print()
        

In [ ]:
print("YEAR   MO",end="")
for dy in range(1,31):
    print(f"     {dy:2d}",end="")
print()
    
for yr,inv in sorted(invtdy.items()):
    for mo,ii in sorted(inv.items()):
        
        print(f"{yr:4d}   {mo:2d} ",end="")
        for dy in range(1,31):
            if dy in ii:
                cnt=ii[dy]
            else:
                cnt=0
            print(f" {cnt:5d} ",end="")
        print()

In [ ]:
histO={}
for line in orig:
    nn = len(line)
    if nn in histO:
        histO[nn]+=1
    else:
        histO[nn]=1

In [ ]:
histO

In [ ]:
histD={}
for nline,lineo in enumerate(orig):
    linen=new[nline]    
    for nn,co in enumerate(lineo):
        cn= linen[nn]
        if co != cn:
            if nn not in [4,5]:
                print(nn,co)
                print(nn,cn)
                print("----")
            
            if nn in histD:
                histD[nn]+=1
            else:
                histD[nn]=1
    if nline > 10000000:
        break
        

In [ ]:
fin2 = open("/home/joe/bic_etl/cdos/business/business/data_transformed/corphist.tsv")
fin1 = open("/home/joe/bic_etl/cdos/business/business/data_transformed/corphist.csv")

reader2 = csv.reader(fin2,delimiter="\t",quoting=3)
reader1 = csv.reader(fin1)

new=[]
nbad=0
nlines=0
histD={}
print("starting")
for row1 in reader1:
   # print(row1)
    nlines+=1
    row2 = next(reader2)
    if nlines > 9360130:
        print(nlines)
        print(row1)
        print(row2)
        print("---------")
    if nlines > 9360140:
        break
        
#"encoding=utf-8-sig"

In [ ]:
9360135-9064135

In [ ]:
fin2 = open("/home/joe/bic_etl/cdos/business/business/data_source/corphist-2.tsv",encoding="latin")

lines = fin2.readlines()


In [ ]:
['20131109385', '20151147043', 'Statement of Trade Name Renewal of a Person other than a Reporting Entity, A Domestic Limited Partnership, a Dissolved or Delinquent Reporting Entity, or a Converted Entity', 'Poudre Valley Chiropractic Center;', '02/27/2015 11:51', '02/27/2015 11:51', '']
['20151315865', '20151315865', 'Statement of Trade Name of a Reporting Entity', 'ISKCON Denver;', '05/10/2015 11:10:59', '05/10/2015 11:10:59', 'ISKCON Denver']
20121098063	20151147042

In [ ]:
for nn in range(295995,296006):
  #  print(nn)
    print(lines[nn][:30])
    line=lines[nn]
    # for mm,c in enumerate(line):
    #     print(mm,c,ord(c))
  #  print("----")

In [ ]:
fin2 = open("/home/joe/bic_etl/cdos/business/business/data_transformed/corphist.tsv")
fin1 = open("/home/joe/bic_etl/cdos/business/business/data_transformed/corphist.csv")
#fin1 = open("Business_Entity_Transaction_History_20240624.csv")


reader2 = csv.reader(fin2,delimiter="\t",quoting=3)
reader1 = csv.reader(fin1)

new=[]
nbad=0
nlines=0
histD={}
print("starting")
for row1 in reader1:
   # print(row1)
    row2 = next(reader2)
  #  print(row2)
    for nn,val1 in enumerate(row1):
        val2=row2[nn]
        if val1 != val2:
            
            if nn not in [4,5] and nbad < 100:
                nbad+=1
                print(nlines,nn,val1,val2,len(val1),len(val2))
                print(row1)
                print(row2)
            if nn in histD:
                histD[nn]+=1
            else:
                histD[nn]=1
    nlines+=1
    if nlines%100000 == 0:
        print(nlines)
    if nbad > 100:
        print("Quitting for bad recrods: ",nbad)
        break
    

In [ ]:
df = pd.read_csv("/home/joe/bic_etl/cdos/business/business/data_transformed/corphist.tsv",delimiter="\t",quoting=3)


In [ ]:
df.columns

In [ ]:
hld = df.groupby(["entityid","transactionid","effectivedate","receiveddate"],as_index=False).count()

In [ ]:
df.shape

In [ ]:
df.drop_duplicates(ignore_index=True).shape

In [ ]:
18627989-18598493

In [ ]:
df1 = pd.read_csv("/home/joe/bic_etl/cdos/business/business/data_source/corphist-1.tsv",delimiter="\t")


In [ ]:
df1.shape

In [ ]:
df1.drop_duplicates(ignore_index=True).shape

In [ ]:
9064134-9039778

In [ ]:
df2 = pd.read_csv("/home/joe/bic_etl/cdos/business/business/data_source/corphist-2.tsv",delimiter="\t",quoting=3,encoding="latin")


In [ ]:
df2.shape

In [ ]:
df2.drop_duplicates(ignore_index=True).shape

In [ ]:
9579699-9574557

In [ ]:
hld.loc[hld["comment"] > 1]

In [ ]:
df['entityid'].value_counts()

In [ ]:
nlines

In [ ]:
histD

In [ ]:
print(new[100000])
print(orig[100000])

In [ ]:

for nline,lineo in enumerate(orig):
    linen=new[nline]    
    for nn,co in enumerate(lineo):
        cn= linen[nn:nn+1]
        if co != cn:
            print(nline,nn,co,cn)
    if nline > 100:
        break
        

In [ ]:
len(latin)

In [ ]:
for nn,lorig in enumerate(orig):
    llatin=latin[nn]
    for no,co in enumerate(lorig):
        cl=llatin[no:no+1]
        if co != cl:
            print(no,co,cl,ord(co),ord(cl))
    
        
print(nn)  

In [ ]:
import unicodedata

def is_english(c):
    print(c,unicodedata.name(c))
    return c.isalpha() and unicodedata.name(c).startswith(('LATIN', 'COMMON'))

def remove_non_english(lst):
    output = []
    for s in lst:
        filtered = filter(is_english, list(s))
        english_str = ''.join(filtered)
        output.append(english_str)
    return output

# initializing list
test_list = ['Gfg', 'Good| ????', "for", '??Geeks???']

# printing original list
print("The original list is : " + str(df1.columns))

# printing result
print("The extracted list : " + str(remove_non_english(df1.columns)))


In [ ]:
xrefs = {
'comment': 'Comment',
'effectivedate': 'Effective Dtm',
'entityid': 'Entity Id',
'historydes': 'History Description',
'name': 'Name',
'receiveddate': 'Received Dtm',
'transactionid': 'Transaction Id'
}




xrefsSrc2Trans={}
chkTrns={}
for trns,src in xrefs.items():
    if src not in xrefsSrc2Trans:
        xrefsSrc2Trans[src]=trns
    else:
        print("DUplicate Source Columns: ",src)
   #     print("Stopping Short")
    #    break
    
    if trns in chkTrns:
        print("Duplicate TRANSFORM Column: ",trns)
    chkTrns[trns]=1
    


In [ ]:
## Create the Dictionary Lookup 
xrefDefs = {}
xrefDefs[w4x4]={}

for src,trans in xrefsSrc2Trans.items():
    xrefDefs[w4x4][src]={
        "xref":trans,
        "desc":defs[trans],
        "notes":"",
        "date":today,
        "source":"cdos"
    }

In [ ]:
fin = open("hld")
for line in fin:
    spl = line.split("\t")
    print(spl)
    print(line)